## ====== Main Code ======

### M0 - Preparation and Installation of the dsm_llm_rag Environment

In [1]:
# Go to the cloned repository scripts path to run the associated steps
# % cd /path/to/xLLM_DSM/scripts/

In [ ]:
# ! pip install langchain==0.0.339 langchain_core openai==1.3.4 chromadb==0.4.17 \
#               langchain_openai langchain_community tiktoken pypdf \
#               anthropic pandas numpy neo4j sentence-transformers networkx \
#               overrides onnxruntime pytz httpx pydantic annotated-types \
#               pydantic-core idna anyio sniffio distro certifi tenacity \
#               langsmith requests-toolbelt zstandard --no-deps

! pip install openai langchain langchain_community \
              langchain_chroma tiktoken pypdf \
              chromadb anthropic pandas numpy \
              neo4j sentence-transformers networkx

### M1 - Import Libraries

Import required libraries to the current runtime.

In [1]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import networkx as nx
import os
import shutil
import glob

from concurrent.futures import ThreadPoolExecutor

# Import LangChain classes with aliases
from langchain.chat_models import ChatOpenAI as LangChainChatOpenAI
from langchain.chat_models import ChatAnthropic as LangChainChatAnthropic
from langchain.chat_models import ChatOllama as LangChainChatOllama
from langchain.chains import RetrievalQA
from langchain.embeddings import OpenAIEmbeddings as LangChainOpenAIEmbeddings
from langchain.embeddings import OllamaEmbeddings as LangChainOllamaEmbeddings
# from langchain_openai.embeddings import OpenAIEmbeddings
from langchain.embeddings import HuggingFaceEmbeddings as SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma
from langchain.schema import HumanMessage, BaseRetriever, Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader

from chromadb.config import Settings
from chromadb.utils import embedding_functions
from chromadb import PersistentClient

import logging
from datetime import datetime

import asyncio
import logging
import os
import shutil
import subprocess
from pathlib import Path
from typing import Any, Dict, List, Optional

import tiktoken
import yaml  # Assuming yaml is used for settings

# GraphRAG specific imports from the notebook
from graphrag.query.context_builder.entity_extraction import EntityVectorStoreKey
from graphrag.query.indexer_adapters import (
    read_indexer_covariates,
    read_indexer_entities,
    read_indexer_relationships,
    read_indexer_reports,
    read_indexer_text_units,
)
from graphrag.query.llm.oai.chat_openai import ChatOpenAI as GraphRAGChatOpenAI
from graphrag.query.llm.oai.embedding import OpenAIEmbedding as GraphRAGOpenAIEmbedding
from graphrag.query.llm.oai.typing import OpenaiApiType
from graphrag.query.structured_search.local_search.mixed_context import (
    LocalSearchMixedContext,
)
from graphrag.query.structured_search.local_search.search import LocalSearch
from graphrag.vector_stores.lancedb import LanceDBVectorStore

In [ ]:
os.environ['OPENAI_API_KEY'] = 'YOUR_OPENAI_API_KEY'
api_key = os.getenv('OPENAI_API_KEY')  # Get API key from environment variable

### M2 - Custom Logger

In [3]:
class CustomLogger:
    """
    Logger class that can be accessed from any other
    class to maintain consistent logging across the application.

    """
    # _instance = None
    # _logger: Optional[logging.Logger] = None

    # def __new__(cls):
    #     if cls._instance is None:
    #         cls._instance = super(CustomLogger, cls).__new__(cls)
    #     return cls._instance
    
    def __init__(self, inference_type: Optional[str] = None, process_name: Optional[str] = None, model_name: Optional[str] = None, timestamp: Optional[str] = None):
        # Only initialize if not already done
        self._logger = None
        self.model_name = model_name
        self.process_name = process_name
        self.inference_type = inference_type
        self.timestamp = timestamp
        self._setup_logger()

    def _setup_logger(self) -> None:
        """Initialize logger with both file and console handlers"""
        # Create logs directory if it does not exist
        log_dir = Path("./logs")
        log_dir.mkdir(parents=True, exist_ok=True)

        # Create log file with timestamp
        model_suffix = f"{self.model_name}" if self.model_name else ""
        log_file = log_dir / f"{self.process_name}_{self.inference_type}_{model_suffix}_{self.timestamp}.log"

        # Initialize logger
        logger_name = f"DSM_{self.process_name}_{self.inference_type}_{model_suffix}"
        self._logger = logging.getLogger(logger_name)
        self._logger.setLevel(logging.INFO)

        # Remove any existing handlers
        if self._logger.handlers:
            self._logger.handlers = []

        # Create file handler
        file_handler = logging.FileHandler(log_file)
        file_handler.setLevel(logging.INFO)

        # Create console handler
        console_handler = logging.StreamHandler()
        console_handler.setLevel(logging.INFO)

        # Create formatter and add it to the handlers
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        file_handler.setFormatter(formatter)
        console_handler.setFormatter(formatter)
        
        # Add handlers to logger
        self._logger.addHandler(file_handler)
        self._logger.addHandler(console_handler)

    def info(self, message: str) -> None:
        """Log info message"""
        if self._logger:
            self._logger.info(message)

    def error(self, message: str, exc_info=False) -> None:
        """Log error message with optional exception info"""
        if self._logger:
            self._logger.error(message, exc_info=exc_info)

    def warning(self, message: str) -> None:
        """Log warning message"""
        if self._logger:
            self._logger.warning(message)

    def debug(self, message: str) -> None:
        """Log debug message"""
        if self._logger:
            self._logger.debug(message)

    def critical(self, message: str) -> None:
        """Log critical message"""
        if self._logger:
            self._logger.critical(message)

    def cleanup(self) -> None:
        """Clean up logger handlers"""
        if self._logger:
            for handler in self._logger.handlers[:]:
                handler.flush()
                handler.close()
                self._logger.removeHandler(handler)


### M3 - System Configuration

In [4]:
@dataclass
class SystemConfig:
    """Configuration class for system details and paths."""
    system_name: str
    concept_name: str
    application_domain: str
    relationship_type: str
    api_keys: Dict[str, str]
    selected_model: str
    embedding_model: str
    inference_type: str # 'llm', 'llm_rag', or 'llm_rag_graph'
    reference_files: Dict[str, List[Dict[str, str]]]
    output_directory: str
    vectorstore_directory: str
    config_path: str= ' '
    graph_config: Optional[Dict[str, str]] = None # Made optional with default None
    ollama_api_key: str = "ollama"
    ollama_api_base: str = "http://localhost:11434/v1"
    ollama_emb_base: str = "http://localhost:11434/api/embeddings"
    check_enable_validation: bool = False
    case: str = "i"
    predicted_components: str = ""
    graphrag_input_dir: str = ""
    reference_types: List[str] = field(default_factory=list)

    @classmethod
    def from_json(cls, config_path: str= 'config.json') -> 'SystemConfig':
        with open(config_path) as file:
            config = json.load(file)

            # Set vectorstore_directory to output_directory if not provided
            if not config.get('vectorstore_directory'):
                config['vectorstore_directory'] = '../data/vectorstore'

            # Only include graph_config if inference_type is 'llm_rag_graph'
            if config.get('inference_type') == 'llm_rag_graph':
                config.pop('graph_config', None)
            
            config['config_path'] = config_path
        return cls(**config)
    
    def set_api_key(self, api_key: str) -> None:
        """Set the appropriate API key based on the selected model"""
        if 'gpt' in self.selected_model.lower():
            self.api_keys['openai_api_key'] = api_key
        elif 'claude' in self.selected_model.lower():
            self.api_keys['claude_api_key'] = api_key
        
        # Add this to ensure the key is available for embeddings
        if 'openai_api_key' not in self.api_keys and 'gpt' in self.selected_model.lower():
            self.api_keys['openai_api_key'] = api_key

### M4 - Invoke VectorDB Class

In [5]:
class VectorStoreManager:
    """Manages vector store operations with persistent storage"""
    def __init__(self, logger: CustomLogger):
        self._vectorstore = {}
        self._chroma_clients = {}
        self._persistent_base = Path.cwd().parent / 'data' / 'vectorstore'
        self._persistent_base.mkdir(parents=True, exist_ok=True)
        self._current_db_path = None

        self.logger = logger
        self.logger.info("VectorStoreManager initialized")

    def set_db_path(self, system_name: str, config: SystemConfig) -> None:
        """
        Set the current database path based on system_name,
        inference_type, and selected_model
        """
        self._current_db_path = Path(f"{self._persistent_base}/{system_name.replace(' ', '')}/{config.inference_type}/{config.embedding_model}/")
        self._current_db_path.mkdir(parents=True, exist_ok=True)
        # print(f"Set database path to: {self._current_db_path}")
        self.logger.info(f"Set database path to: {self._current_db_path}")

    def get_db_path(self, system_name: str = None) -> Path:
        """Get persistent path for system"""
        if self._current_db_path is None:
            raise ValueError("Database path not set. Call set_db_path() first.")
        return self._current_db_path

    def cleanup_system_db(self, system_name: str) -> None:
        """Clean up persistent storage"""
        if self._current_db_path is None:
            raise ValueError("Database path not set. Call set_db_path() first.")

        self.logger.info(f"Cleaning up system at path: {self._current_db_path}")

        try:
            # Clean up client if it exists for this specific path
            client_key = f"{system_name}_{self._current_db_path}"
            if client_key in self._chroma_clients:
                client = self._chroma_clients[client_key]
                try:
                    client.reset()
                except Exception as e:
                    self.logger.warning(f"Error resetting client: {e}")
                del self._chroma_clients[client_key]
                self.logger.info(f"Reset and removed client for path: {self._current_db_path}")

            # Clean up vectorstore if it exists
            if client_key in self._vectorstore:
                del self._vectorstore[client_key]
                self.logger.info(f"Removed vectorstore reference for path: {self._current_db_path}")

            # Clean up directory with proper permissions handling
            if self._current_db_path.exists():
                try:
                    shutil.rmtree(self._current_db_path)
                    self.logger.info(f"Deleted directory: {self._current_db_path}")
                except PermissionError:
                    import os
                    # Make files writable before deletion
                    for root, dirs, files in os.walk(self._current_db_path):
                        for dir in dirs:
                            os.chmod(os.path.join(root, dir), 0o777)
                        for file in files:
                            os.chmod(os.path.join(root, file), 0o777)
                    shutil.rmtree(self._current_db_path)
                    
            self._current_db_path.mkdir(parents=True, exist_ok=True)
            self.logger.info(f"Re-Created new directory: {self._current_db_path}")
            
        except Exception as e:
            self.logger.error(f"Error during cleanup: {e}")

    def get_system_db(self, system_name: str, config: SystemConfig) -> Optional[Chroma]:
        """Get vectorstore for a specific system"""
        persistent_path = self.get_db_path(system_name)
        # print(f"Looking for vectorstore at: {persistent_path}")
        # print(f"Checking if path exists: {persistent_path.exists()}")
        self.logger.info(f"Looking for vectorstore at: {persistent_path}")
        self.logger.info(f"Checking if path exists: {persistent_path.exists()}")

        try:
            if system_name not in self._chroma_clients:
                # Interface object to access the ChromaDB database,
                self._chroma_clients[system_name] = PersistentClient(
                    path=str(persistent_path),
                    settings=Settings(
                        anonymized_telemetry=False,
                        allow_reset=True,
                        is_persistent=True
                    )
                )
                # print(f"Initialized PersistentClient for {system_name}")
                self.logger.info(f"Initialized PersistentClient for {system_name}")

            client = self._chroma_clients[system_name]
            if 'nomic' in config.embedding_model.lower():
                embedding_function = embedding_functions.OllamaEmbeddingFunction(
                    model_name=config.embedding_model,
                    url=config.ollama_emb_base
                )
            else:
                embedding_function = embedding_functions.OpenAIEmbeddingFunction(
                    api_key=os.getenv('OPENAI_API_KEY'),
                    model_name=config.embedding_model
                )

            try:
                # Try to get existing collection
                collection = client.get_or_create_collection(
                    name=f"{system_name}_collection",
                    embedding_function=embedding_function  # Pass embedding function here
                )
                # print(f"Collection size after initialization: {collection.count()}")
                self.logger.info(f"Collection size after initialization: {collection.count()}")
                # print(f"Successfully retrieved collection for {system_name}.")
                self.logger.info(f"Successfully retrieved collection for {system_name}.")
            except Exception as e:
                # print(f"Error getting collection: {str(e)}")
                self.logger.error(f"Error getting collection: {str(e)}")
                return None

            vectorstore = Chroma(
                client=client,
                collection_name=f"{system_name}_collection",
                embedding_function=embedding_function  # Pass the same embedding function here
            )

            self._vectorstore[system_name] = vectorstore
            return vectorstore

        except Exception as e:
            # print(f"Error accessing ChromaDB: {str(e)}")
            self.logger.error(f"Error accessing ChromaDB: {str(e)}")
            return None

    def verify_vectorstore(self, system_name: str):
        vector_manager = VectorStoreManager()
        vectorstore = vector_manager.get_system_db(system_name)

        if vectorstore:
            try:
                # Test a simple query
                results = vectorstore.similarity_search("test", k=1)
                # print(f"\nVectorstore verification for {system_name}:")
                self.logger.info(f"\nVectorstore verification for {system_name}:")
                # print(f"Query successful: {len(results) > 0}")
                self.logger.info(f"Query successful: {len(results) > 0}")
                # print(f"Number of documents: {len(results)}")
                self.logger.info(f"Number of documents: {len(results)}")
            except Exception as e:
                # print(f"Error querying vectorstore: {str(e)}")
                self.logger.error(f"Error querying vectorstore: {str(e)}")
        else:
            # print(f"Could not get vectorstore for {system_name}")
            self.logger.error(f"Could not get vectorstore for {system_name}")

    # Add this debugging code after the reference_files dictionary is created in generate_system_db
    def debug_system_files(self,system_name: str, base_path: str = "../data/use_cases_large"):
        system_path = Path(base_path) / str(system_name) / "reference_pdfs"
        # print(f"\nDebugging {system_name} files:")
        self.logger.info(f"\nDebugging {system_name} files:")
        # print(f"System path exists: {system_path.exists()}")
        self.logger.info(f"System path exists: {system_path.exists()}")
        # print(f"System path: {system_path}")
        self.logger.info(f"System path: {system_path}")
        # print("\nFiles in directory:")
        self.logger.info("\nFiles in directory:")
        if system_path.exists():
            for file in system_path.glob("*"):
                # print(f"- {file.name}")
                self.logger.info(f"- {file.name}")
                # print(f"  File size: {file.stat().st_size} bytes")
                self.logger.info(f"  File size: {file.stat().st_size} bytes")
                # print(f"  File readable: {os.access(str(file), os.R_OK)}")
                self.logger.info(f"  File readable: {os.access(str(file), os.R_OK)}")

    def initialize_system_db(self,
                             system_name: str,
                             reference_files: Dict[str, List[Dict]],
                             config: SystemConfig) -> Dict[str, Any]:
        """Initialize vectorstore for a specific system"""
        if self._current_db_path is None:
            raise ValueError("Database path not set. Call set_db_path() first.")

        # print(f"Initializing vectorstore for system: {system_name}")
        self.logger.info(f"Initializing vectorstore for system: {system_name}")
        # print(f"Using path: {self._current_db_path}")
        self.logger.info(f"Using path: {self._current_db_path}")
        documents = []

        try:
            client = PersistentClient(
                path=str(self._current_db_path),
                settings=Settings(
                    anonymized_telemetry=False,
                    allow_reset=True,
                    is_persistent=True
                )
            )
            client_key = f"{system_name}_{self._current_db_path}"
            self._chroma_clients[client_key] = client
            try:
                if 'nomic' in config.embedding_model.lower():
                    self.logger.info("Initialize ollama embedding function")
                    embedding_function = embedding_functions.OllamaEmbeddingFunction(
                        model_name=config.embedding_model,
                        url="http://localhost:11434/api/embeddings"
                    )
                else:
                    embedding_function = embedding_functions.OpenAIEmbeddingFunction(
                        api_key=os.getenv('OPENAI_API_KEY'),
                        model_name=config.embedding_model
                    )
            except Exception as e:
                self.logger.error(f"Error initializing embedding function: {str(e)}")
                raise

            # Process documents
            for r_type, refs in reference_files.items():
                # print(f"Processing {r_type} documents...")
                self.logger.info(f"Processing {r_type} documents...")
                for ref in refs:
                    try:
                        loader = PyPDFLoader(ref['path'])
                        docs = loader.load()
                        for doc in docs:
                            doc.metadata.update({
                                'r_type': r_type,
                                'system': system_name,
                                'source': ref['path']
                            })
                        documents.extend(docs)
                    except Exception as e:
                        # print(f"Error loading {ref['path']}: {e}")
                        self.logger.error(f"Error loading {ref['path']}: {e}")

            if documents:
                text_splitter = RecursiveCharacterTextSplitter(
                    chunk_size=1200,
                    chunk_overlap=300
                )
                splits = text_splitter.split_documents(documents)
                # Log document processing progress
                self.logger.info(f"Processing {len(splits)} text chunks...")

                try:
                    collection = client.get_or_create_collection(
                        name=f"{system_name}_collection",
                        metadata={
                            "system": system_name,
                            "model": config.selected_model,
                            "inference_type": config.inference_type
                        },
                        embedding_function=embedding_function
                    )

                    # Add documents in batches
                    texts = [split.page_content for split in splits]
                    metadatas = [{
                        'r_type': split.metadata['r_type'],
                        'system': split.metadata['system'],
                        'source': split.metadata['source'],
                        'page': split.metadata.get('page', 0)
                    } for split in splits]
                    ids = [f"{system_name}_{i}" for i in range(len(splits))]

                    batch_size = 100
                    for i in range(0, len(texts), batch_size):
                        try:
                            end_idx = min(i + batch_size, len(texts))
                            self.logger.info(f"Processing batch {i//batch_size + 1} of {len(texts)//batch_size + 1}")
                                                        
                            collection.add(
                                documents=texts[i:end_idx],
                                metadatas=metadatas[i:end_idx],
                                ids=ids[i:end_idx]
                            )
                            self.logger.info(f"Successfully added batch {i//batch_size + 1}")
                        except Exception as e:
                            self.logger.error(f"Error adding batch {i//batch_size + 1}: {str(e)}")
                            raise

                    return {
                        'vectorstore': collection,
                        'documents': documents,
                        'document_count': len(documents)
                    }

                except Exception as e:
                    self.logger.error(f"Error in collection processing: {str(e)}")
                    raise
        
        except Exception as e:
            # print(f"Error creating vectorstore: {e}")
            self.logger.error(f"Error creating vectorstore: {e}")
            if self._current_db_path.exists():
                shutil.rmtree(self._current_db_path)
            raise

### M5a - Generate VectorStore DB (First Invoke VectorDB Class) - DO NOT RUN IF VECTORDB ALREADY EXISTS

In [6]:
def generate_system_db(system_name: str, 
                      config: SystemConfig,
                      api_key: str,
                      base_path: str = "../data/use_cases_large") -> None:
    """Generate vectorstore DB for a system using R-type files"""
    # Validate inference type
    if config.inference_type not in ['llm_rag', 'llm_rag_graph']:
        raise ValueError(f"Invalid inference type: {config.inference_type}. Must be 'llm_rag' or 'llm_rag_graph'")

    # Make paths explicit
    document_path = Path(base_path) / system_name / "reference_pdfs"
    print(f"\nProcessing system: {system_name}")
    print(f"Input path: {document_path}")
    print(f"Path exists: {document_path.exists()}")
    
    logger = CustomLogger("DB_generation", config.embedding_model)
    # Construct reference files dictionary by R-type
    reference_files = {}
    for pdf_file in document_path.glob("*.pdf"):
        # Look for R1, R2, or R3 in the filename
        for r_type in ['R1', 'R2', 'R3']:
            if r_type in pdf_file.name:
                if r_type not in reference_files:
                    reference_files[r_type] = []
                reference_files[r_type].append({'path': str(pdf_file)})
                break  # Stop after finding first match
    
    print(f"\nFound reference files: {reference_files}")
    
    try:
        # Initialize VectorStoreManager
        vector_manager = VectorStoreManager(logger)
        vector_manager.set_db_path(system_name, config)
        print(f"Vectorstore will be saved to: {vector_manager.get_db_path()}")

        # Clean existing DB
        vector_manager.cleanup_system_db(system_name)
        
        # Add a small delay to ensure cleanup is complete
        time.sleep(10)
        
        # Update config with reference files and API key
        config.reference_files = reference_files
        config.set_api_key(api_key)
        
        # Initialize new DB with error handling
        try:
            result = vector_manager.initialize_system_db(
                system_name=system_name,
                reference_files=reference_files,
                config=config
            )
            
            if result:
                print(f"\nSuccessfully created vectorstore for {system_name}")
                print(f"Documents loaded: {result['document_count']}")
                print(f"Vectorstore location: {vector_manager.get_db_path()}")
            else:
                print(f"\nFailed to create vectorstore for {system_name}")
                
        except Exception as e:
            print(f"Error initializing vectorstore: {str(e)}")
            vector_manager.cleanup_system_db(system_name)
            raise
            
    except Exception as e:
        print(f"Error processing system {system_name}: {str(e)}")
        vector_manager.cleanup_system_db(system_name)

In [7]:
def create_model_configs(system_name: str,
                         embedding_model: str,
                         api_key: str,
                         vectorstore_dir: Path) -> List[SystemConfig]:
    """Create configurations for different inference types and their supported models"""
    config_templates = {
        'llm_rag': [
            # 'nomic-embed-text',
            'text-embedding-3-small',
            # 'gpt-4o-2024-11-20',
            # 'mixtral:8x22b',
            # 'llama3.3:70b',
            # 'deepseek-r1:14b',
        ],
        'llm_rag_graph': [
            # 'nomic-embed-text',
            'text-embedding-3-small',
            # 'gpt-4o-2024-11-20',
            # 'mixtral:8x22b',
            # 'llama3.3:70b',
            # 'deepseek-r1:14b',
        ]
    }

    configs = []
    for inference_type, models in config_templates.items():
        for model in models:
            config = SystemConfig(
                system_name=system_name,
                selected_model=model,
                embedding_model=embedding_model,
                api_keys={'openai_api_key': api_key},
                output_directory=str(vectorstore_dir / inference_type / model),
                vectorstore_directory=str(vectorstore_dir),
                application_domain='engineering',
                relationship_type='functional',
                inference_type=inference_type,
                reference_files=[],  # This will be populated in generate_system_db
                graph_config={} if inference_type == 'llm_rag_graph' else None
            )
            configs.append(config)

    return configs

In [ ]:
# Double click to expand -> Generate VectorStore DB for specific system
# Reset and restart the notebook for generating new vectorstores.
# Otherwise you will get an error as "Error creating vectorstore: attempt to write a readonly database"
target_system = "CubeSat"  # Set to None to process all systems otherwise set to system name e.g., "CubeSat", "CuttingSystem", etc.
base_path = Path("../data/use_cases_large")
embedding_model = "text-embedding-3-small" #"nomic-embed-text" #

# Single system processing
if target_system:
    print(f"\nProcessing single system: {target_system}")
    print("-" * 50)
    system_dir = base_path / target_system
    vectorstore_dir = Path("../data/vectorstores") / embedding_model / target_system
    
    if not system_dir.exists():
        print(f"System directory not found: {system_dir}")
    else:
        configs = create_model_configs(target_system, embedding_model, api_key, vectorstore_dir)
        for config in configs:
            try:
                print(f"\nProcessing {target_system}")
                print(f"Inference type: {config.inference_type}")
                print(f"Model: {config.selected_model}")
                generate_system_db(target_system, config, api_key)
                print(f"End of processing {target_system}")
                time.sleep(10)  # Delay between runs
            except Exception as e:
                print(f"Failed to process {target_system}: {str(e)}")
            finally:    
                print("-" * 50)

# Multiple systems processing
else:
    systems = [folder.name for folder in base_path.iterdir() if folder.is_dir()]
    print(f"Found {len(systems)} systems to process:")
    print("\n".join(f"- {system}" for system in systems))

    for system in systems:
        system_dir = base_path / system
        vectorstore_dir = Path("../data/vectorstores") / system
        
        if not system_dir.exists():
            print(f"System directory not found: {system_dir}")
            continue

        configs = create_model_configs(system, api_key, vectorstore_dir)
        for config in configs:
            try:
                print(f"\nProcessing {system}")
                print(f"Inference type: {config.inference_type}")
                print(f"Model: {config.selected_model}")
                generate_system_db(system, config, api_key)
                # print(f"Successfully processed {system}")
                time.sleep(10)  # Delay between runs
            except Exception as e:
                print(f"Failed to process {system}: {str(e)}")
                continue
            finally:
                print("-" * 50)

### M5b- Preparing for graphRAG (Initialize, Autotune, Index)

We provided some of the required files inside ../{graphRAG_directory}/ragtest/input. You can use example structure with the following code for testing an output first for verification of the process following by your own implementation with the same workflow.

#### Initialize

In [ ]:
! bash graphrag_runner.sh init --root ../ragtest/{experiment_path} #../ragtest/auto_gpt4tp_r1_r2_te3s

#### Autotune

#### Indexing (Not necessary if you already completed it)

In [ ]:
# Make sure that you are in the scripts folder
# I recommend to run the following command in a terminal that you can attach in case of network issues
# tmux is a good tool for this purpose.
! bash graphrag_runner.sh index --root ../ragtest/{experiment_path} #../ragtest/auto_gpt4tp_r1_r2_te3s

### M6 - Inference Classes (LLM, RAG, GraphRAG) ###

In [6]:
class InferenceBase(ABC):
    """Base class for different inference approaches"""
    def __init__(self, config: SystemConfig):
        self.config = config
        self.llm = self._initialize_llm()

    def _initialize_llm(self):
        """Initialize the language model based on configuration"""
        # Import needed modules
        from graphrag.query.llm.oai.typing import OpenaiApiType
        # Helper to get logger if it exists
        log_func = getattr(self, 'logger', None)

        # If we're in GraphRAG mode, prioritize GraphRAG's LLM handlers
        if self.config.inference_type == 'llm_rag_graph':
            model_name_lower = self.config.selected_model.lower()

            # Log entry point for debugging
            if log_func and hasattr(log_func, 'info'):
                log_func.info(f"Initializing LLM for GraphRAG. Model: '{self.config.selected_model}', Type: '{self.config.inference_type}'")
            else:
                 print(f"INFO: Initializing LLM for GraphRAG. Model: '{self.config.selected_model}', Type: '{self.config.inference_type}'")

            if 'gpt' in model_name_lower:
                # Use GraphRAG's OpenAI handler for GPT models
                if log_func and hasattr(log_func, 'info'):
                    log_func.info(f"Using GraphRAGChatOpenAI for GPT model: {self.config.selected_model}")
                else:
                    print(f"INFO: Using GraphRAGChatOpenAI for GPT model: {self.config.selected_model}")
                return GraphRAGChatOpenAI(
                    model=self.config.selected_model,
                    api_key=self.config.api_keys.get('openai_api_key', os.getenv('OPENAI_API_KEY')),
                    api_type=OpenaiApiType.OpenAI
                    # Add temperature=0 if GraphRAGChatOpenAI supports it and it's desired
                )
            elif 'ollama' in model_name_lower:
                # Try using GraphRAGChatOpenAI pointed at Ollama endpoint
                model_parts = self.config.selected_model.split(':')
                if len(model_parts) >= 2:
                    model_identifier = ":".join(model_parts[1:])
                    if log_func and hasattr(log_func, 'info'):
                        log_func.info(f"Attempting GraphRAGChatOpenAI for Ollama model: {model_identifier} at {self.config.ollama_api_base}")
                    else:
                        print(f"INFO: Attempting GraphRAGChatOpenAI for Ollama model: {model_identifier} at {self.config.ollama_api_base}")

                    try:
                        # Assume GraphRAGChatOpenAI accepts api_base and temperature
                        # Verify signature if possible; this assumes compatibility
                        # Setting temperature=0 explicitly
                        return GraphRAGChatOpenAI(
                            model=model_identifier,
                            api_base=self.config.ollama_api_base,
                            api_type=OpenaiApiType.OpenAI, # Assuming Ollama endpoint mimics OpenAI structure
                            # deployment_name=model_identifier, # May be needed depending on GraphRAGChatOpenAI version
                        )
                    except TypeError as te:
                        # Handle if GraphRAGChatOpenAI signature doesn't accept api_base etc.
                        if log_func and hasattr(log_func, 'error'):
                             log_func.error(f"GraphRAGChatOpenAI init failed for Ollama (likely signature mismatch): {te}. Trying LangChainChatOllama fallback.")
                        else:
                             print(f"ERROR: GraphRAGChatOpenAI init failed for Ollama (likely signature mismatch): {te}. Trying LangChainChatOllama fallback.")
                        # Fallback to LangChainChatOllama
                        return GraphRAGChatOpenAI(
                            model=model_identifier,
                            base_url=self.config.ollama_api_base,
                        )
                    except Exception as e:
                         # Catch other potential errors during init
                        if log_func and hasattr(log_func, 'error'):
                             log_func.error(f"Unexpected error initializing GraphRAGChatOpenAI for Ollama: {e}. Trying LangChainChatOllama fallback.")
                        else:
                             print(f"ERROR: Unexpected error initializing GraphRAGChatOpenAI for Ollama: {e}. Trying LangChainChatOllama fallback.")
                        return GraphRAGChatOpenAI(
                            model=model_identifier,
                            base_url=self.config.ollama_api_base,
                        )
                else:
                    # Handle unexpected Ollama format
                    if log_func and hasattr(log_func, 'warning'):
                        log_func.warning(f"Ollama model name format unexpected: {self.config.selected_model}. Trying LangChainChatOllama with full name.")
                    else:
                        print(f"WARNING: Ollama model name format unexpected: {self.config.selected_model}. Trying LangChainChatOllama with full name.")
                    return LangChainChatOllama(
                        model=self.config.selected_model,
                        base_url=self.config.ollama_api_base,
                        temperature=0
                    )
            else:
                # If neither GPT nor Ollama in GraphRAG mode, raise error
                raise ValueError(f"Unsupported model for GraphRAG inference: {self.config.selected_model}")

        # Otherwise use LangChain's models for standard LLM/RAG
        else:
            model_name_lower = self.config.selected_model.lower()
            if log_func and hasattr(log_func, 'info'):
                 log_func.info(f"Initializing LLM for LLM/RAG. Model: '{self.config.selected_model}', Type: '{self.config.inference_type}'")
            else:
                 print(f"INFO: Initializing LLM for LLM/RAG. Model: '{self.config.selected_model}', Type: '{self.config.inference_type}'")

            if 'gpt' in model_name_lower:
                return LangChainChatOpenAI(
                    model_name=self.config.selected_model,
                    openai_api_key=self.config.api_keys.get('openai_api_key', os.getenv('OPENAI_API_KEY')),
                    temperature=0
                )
            elif 'claude' in model_name_lower:
                # This part would need langchain_anthropic installed
                try:
                    return LangChainChatAnthropic(
                        model_name=self.config.selected_model,
                        anthropic_api_key=self.config.api_keys.get('claude_api_key', os.getenv('ANTHROPIC_API_KEY')),
                        temperature=0
                    )
                except NameError:
                     raise ImportError("LangChainChatAnthropic not available. Please install langchain-anthropic.")
                except Exception as e:
                     raise ValueError(f"Error initializing Anthropic model: {e}")

            elif 'ollama' in model_name_lower:
                model_parts = self.config.selected_model.split(':')
                if len(model_parts) >= 2:
                    model_identifier = ":".join(model_parts[1:])
                    return LangChainChatOllama(
                        model=model_identifier,
                        base_url=self.config.ollama_api_base,
                        temperature=0
                    )
                else:
                    if log_func and hasattr(log_func, 'warning'):
                        log_func.warning(f"Ollama model name format unexpected for LLM/RAG: {self.config.selected_model}. Using full name.")
                    else:
                        print(f"WARNING: Ollama model name format unexpected for LLM/RAG: {self.config.selected_model}. Using full name.")
                    return LangChainChatOllama(
                        model=self.config.selected_model,
                        base_url=self.config.ollama_api_base,
                        temperature=0
                    )
            else:
                 # Fallback for standard LLM/RAG if model type not recognized
                 raise ValueError(f"Unsupported model for {self.config.inference_type} inference: {self.config.selected_model}")

    @abstractmethod
    def query(self, prompt: str) -> str:
        """Query method to be implemented by specific inference approaches"""
        pass

class LLMInference(InferenceBase):
    """Basic LLM inference without additional context"""
    def __init__(self, config: SystemConfig, logger: CustomLogger):
        super().__init__(config)
        self.logger = logger
        self.logger.info("DEBUG: Initializing LLMInference.")

    def query(self, prompt: str) -> str:
        response = self.llm([HumanMessage(content=prompt)])
        self.logger.info(f"LLM Response: {response.content}")
        return response.content

class RAGInference(InferenceBase):
    def __init__(self, config: SystemConfig, logger: CustomLogger):
        super().__init__(config)
        self.logger = logger
        self.logger.info("DEBUG: Initializing RAGInference.")

        # Get the reference type from the config filename
        self.r_type = [r_type for r_type in ['R1', 'R2', 'R3'] if r_type \
                       in config.reference_files]
        self.logger.info(f"Using R-type: {self.r_type}")

        # ####
        # # initialize the embeddings for langChain
        # config.embedding_model = "text-embedding-3-large"

        self.logger.info(f"Using embedding model: {config.embedding_model}")
        # Initialize embeddings based on model type
        if 'nomic' in config.embedding_model.lower():
            self.embeddings = LangChainOllamaEmbeddings(
                model=config.embedding_model,
                base_url="http://localhost:11434"
            )
        else:
            self.embeddings = LangChainOpenAIEmbeddings(
                model=config.embedding_model,
                openai_api_key=os.getenv('OPENAI_API_KEY'),
            )

        vector_manager = VectorStoreManager(logger)
        # db_system_name = config.embedding_model.replace(" ", "")
        db_system_name = config.system_name.replace(" ", "")
        print(f"Database system name: {config.system_name}")

        # Set the database path before trying to access it
        # PersistentPath is already set default to {project_root}/data/vectorstores
        vector_manager.set_db_path(config.system_name, config)

        # # Create Chroma instance with LangChain embeddings
        client = PersistentClient(
            path=str(vector_manager.get_db_path(config.system_name)),
            settings=Settings(anonymized_telemetry=False,
                              allow_reset=True,
                              is_persistent=True
            )
        )

        try:
            collection = client.get_or_create_collection(name=f"{db_system_name}_collection")
            count = collection.count()
            self.logger.info(f"Collection count: {count}")

            self.vectorstore = Chroma(
                client=client,
                collection_name=f"{db_system_name}_collection",
                embedding_function=self.embeddings
            )
        
        except Exception as e:
            self.logger.warning(f"Error creating vectorstore: {str(e)}")
            raise ValueError(f"Vectorstore not found for system: {config.system_name}")

        # Initialize the retriever with correct embedding function
        self.retriever = self.vectorstore.as_retriever(
            search_type = 'similarity',
            search_kwargs = {
                'k': 4,
                'filter': {'r_type': {'$in': self.r_type}}
            }
        )

        self.qa_chain = RetrievalQA.from_chain_type(
            self.llm,
            retriever=self.retriever,
            return_source_documents=True
        )

    def query(self, prompt: str) -> str:
        try:
            result = self.qa_chain({'query': prompt})

            # Print which documents were used
            self.logger.info(f"Number of documents retrieved: {len(result['source_documents'])}")
            self.logger.info("Documents used for this query:")

            # Print the source documents
            for doc in result['source_documents']:
                self.logger.info(f"File: {doc.metadata['source']}")
                self.logger.info(f"Page: {doc.metadata['page']}")
                self.logger.info(f"Content: {doc.page_content[:200]} ...") # First 200 characters
            
            # Get the raw response
            response = result['result']
            self.logger.info(f"Raw Response: {result['result']}")

            return response

        except Exception as e:
            self.logger.error(f"Error in RAG query: {str(e)}")
            return "Error in processing query"

class GraphRAGInference(InferenceBase):
    def __init__(self, config, logger):
        super().__init__(config)
        self.logger = logger
        self.logger.info("DEBUG: Initializing GraphRAGInference.")
        
        # Format model name for path construction
        model_name = config.selected_model.lower()
        
        # Specific model name transformations for directory name
        if "gpt-4-turbo-preview" in model_name:
            model_name_formatted = "gpt4tp"
        elif "gpt-4o-" in model_name:
            model_name_formatted = "gpt4o"
        elif "deepseek-r1:14b-5k" in model_name:
            model_name_formatted = "deepseek_r1_14b_5k"
        elif "mixtral:8x22b-6k" in model_name:
            model_name_formatted = "mixtral_8x22b_6k"
        elif "llama3.3:70b-8k" in model_name:
            model_name_formatted = "llama3_3_70b_8k"
        else:
            # Generic formatting for other models
            model_name_formatted = model_name.replace("-", "_").replace(":", "_").replace(".","_")
        
        # Get reference types
        reference_types = "_".join(config.reference_types) if config.reference_types else ""
        
        # Extract embedding suffix
        if "text-embedding-3-small" in config.embedding_model:
            embedding_suffix = "te3s"
        elif "text-embedding-3-large" in config.embedding_model:
            embedding_suffix = "te3l"
        else:
            embedding_suffix = config.embedding_model.split("-")[-1] 
            
        # Build directory name
        if reference_types:
            graph_dir_name = f"auto_{model_name_formatted}_{reference_types}-{embedding_suffix}"
        else:
            graph_dir_name = f"auto_{model_name_formatted}-{embedding_suffix}"
        
        self.logger.info(f"Model name formatted: {graph_dir_name}")          
        # System acronym
        system_acronym = "PS" if "power" in config.concept_name.lower() else "CS"
        
        # Use relative path as requested
        self.graph_dir = f"../data/ragtest/{system_acronym}/{graph_dir_name}"
        
        self.logger.info(f"Using GraphRAG directory: {self.graph_dir}")
        
        # Initialize search engine components
        try:
            # Set up paths to indexed data
            self.entity_table = f"{self.graph_dir}/output/create_final_nodes.parquet"
            self.entity_embedding_table = f"{self.graph_dir}/output/create_final_entities.parquet"
            self.relationship_table = f"{self.graph_dir}/output/create_final_relationships.parquet"
            self.report_table = f"{self.graph_dir}/output/create_final_community_reports.parquet"
            self.text_unit_table = f"{self.graph_dir}/output/create_final_text_units.parquet"
            # self.covariate_table = f"{self.graph_dir}/output/create_final_covariates.parquet"
            self.lancedb_uri = f"{self.graph_dir}/output/lancedb"
            
            # Initialize components needed for search
            self.community_level = 2  # Default community level
            
            # The rest will be initialized during query
            self.logger.info("GraphRAG inference initialized successfully")
            
        except Exception as e:
            self.logger.error(f"Error initializing GraphRAG: {str(e)}")
            raise
    
    def query(self, prompt: str) -> str:
        try:
            # Lazy initialization of search components
            if not hasattr(self, 'search_engine'):
                # Set up search engine components
                try:
                    self.logger.info("Setting up search engine for GraphRAG")

                    # --- Load DataFrames ---
                    self.logger.info("Reading Parquet files...")
                    nodes_df_raw = pd.read_parquet(self.entity_table)
                    entities_df_raw = pd.read_parquet(self.entity_embedding_table)
                    relationships_df_raw = pd.read_parquet(self.relationship_table)
                    reports_df_raw = pd.read_parquet(self.report_table)
                    text_units_df_raw = pd.read_parquet(self.text_unit_table)
                    # Load covariates - requires uncommenting the path definition in __init__!
                    # self.covariate_table = f"{self.graph_dir}/output/create_final_covariates.parquet" # In __init__
                    covariates_df_raw = None # Default to None
                    # Check if covariate_table attribute exists and is defined
                    if hasattr(self, 'covariate_table') and self.covariate_table:
                        try:
                            covariates_df_raw = pd.read_parquet(self.covariate_table)
                            self.logger.info("Covariates file read successfully.")
                        except FileNotFoundError:
                            self.logger.warning(f"Covariates file not found ({self.covariate_table}). Proceeding without covariates.")
                        except Exception as e: # Catch other potential errors reading the file
                            self.logger.warning(f"Could not read covariates file ({self.covariate_table}): {e}. Proceeding without covariates.")
                    else:
                        self.logger.info("Covariate table path not configured. Proceeding without covariates.")

                    self.logger.info("Parquet files read.")

                    # --- Process DataFrames ---
                    self.logger.info("Processing indexer data...")
                    entities = read_indexer_entities(nodes_df_raw, entities_df_raw, self.community_level)
                    relationships = read_indexer_relationships(relationships_df_raw)
                    reports = read_indexer_reports(reports_df_raw, final_nodes=nodes_df_raw, community_level=self.community_level)
                    text_units = read_indexer_text_units(text_units_df_raw)
                    covariates = read_indexer_covariates(covariates_df_raw) if covariates_df_raw is not None else None
                    self.logger.info("Indexer data processed.")

                    # --- Initialize Tokenizer and Embedder ---
                    self.logger.info("Initializing tokenizer and embedder...")
                    token_encoder = tiktoken.get_encoding("cl100k_base")
                    text_embedder = GraphRAGOpenAIEmbedding( # Use GraphRAG specific embedder
                        api_key=self.config.api_keys.get('openai_api_key', os.getenv('OPENAI_API_KEY')),
                        model=self.config.embedding_model, # Use model from config
                        # deployment_name=self.config.embedding_model, # May be needed for Azure
                        max_retries=20,
                    )
                    self.logger.info("Tokenizer and embedder initialized.")

                    # --- Initialize Vector Store ---
                    self.logger.info("Initializing vector store...")
                    # Assuming the collection name matches indexing output, adjust if needed
                    # collection_name = f"vector_store_comm{self.community_level}"
                    # description_embedding_store = LanceDBVectorStore(
                    #     uri=self.lancedb_uri,
                    #     collection_name=collection_name,
                    #     embedding=text_embedder # Pass the embedder
                    # )
                    description_embedding_store = LanceDBVectorStore(
                        collection_name="default-entity-description",
                    )
                    description_embedding_store.connect(db_uri=self.lancedb_uri)
                    # description_embedding_store.connect() # Connect if needed by your LanceDB version/setup
                    # self.logger.info(f"Vector store initialized (Collection: {collection_name}).")

                    # --- Initialize Context Builder ---
                    self.logger.info("Initializing context builder...")
                    # Ensure all required arguments are passed based on the library version you use
                    context_builder = LocalSearchMixedContext(
                        community_reports=reports,
                        text_units=text_units,
                        entities=entities,
                        relationships=relationships,
                        covariates=covariates, # Pass the processed covariates DataFrame (or None)
                        entity_text_embeddings=description_embedding_store, # Pass the vector store here
                        embedding_vectorstore_key=EntityVectorStoreKey.ID,  # Use ID as per example, ensure it matches how store was built
                        text_embedder=text_embedder, # Pass the embedder
                        token_encoder=token_encoder, # Pass the encoder
                    )
                    self.logger.info("Context builder initialized.")

                    # --- Define Params ---
                    # Adjust these based on your model and desired behavior
                    local_context_params = {
                        "text_unit_prop": 0.5,
                        "community_prop": 0.1,
                        "conversation_history_max_turns": 5,
                        "conversation_history_user_turns_only": True,
                        "top_k_mapped_entities": 10,
                        "top_k_relationships": 10,
                        "include_entity_rank": True,
                        "include_relationship_weight": True,
                        "include_community_rank": False, # Set to False if community rank isn't used/available
                        "return_candidate_context": False,
                        "embedding_vectorstore_key": EntityVectorStoreKey.ID, # Use ID here too
                        "max_tokens": 65536, # Adjust based on LLM context window (e.g., 12k for gpt-4-turbo, 5k for some others)
                    }
                    llm_params = {
                        "max_tokens": 6144, # Max tokens for the LLM response generation
                        "temperature": 0.0,
                        # Add other LLM params like 'top_p', 'frequency_penalty' if needed by GraphRAGChatOpenAI or your LLM
                    }
                    self.logger.info(f"Context Params: {local_context_params}")
                    self.logger.info(f"LLM Params: {llm_params}")

                    # --- Initialize Search Engine ---
                    self.logger.info("Initializing LocalSearch engine...")
                    # Note: Using the self.llm initialized in the base class (GraphRAGChatOpenAI)
                    self.search_engine = LocalSearch(
                        llm=self.llm,
                        context_builder=context_builder,
                        token_encoder=token_encoder, # Pass encoder
                        llm_params=llm_params, # Pass LLM params
                        context_builder_params=local_context_params, # Pass context params
                        response_type="multiple paragraphs", # Define desired output format (e.g., "single paragraph", "list")
                    )
                    self.logger.info("LocalSearch engine initialized.") # Log message confirms init call finished

                    # +++ Add these lines for debugging +++
                    self.logger.info(f"Type of self.search_engine after init: {type(self.search_engine)}")
                    if self.search_engine is None:
                        self.logger.error("!!! Critical: self.search_engine is None immediately after initialization !!!")
                        # Optionally raise a more specific error here if needed
                        raise TypeError("LocalSearch initialization failed, returned None")
                    # +++ End of added lines +++

                    # --- Perform Search (First Time) ---
                    self.logger.info(f"Performing initial search with prompt: {prompt[:100]}...")
                    # Use search for synchronous, asearch for async
                    result = self.search_engine.search(query=prompt) # Error might be inside this call
                    self.logger.info(f"Initial search completed. Result response: {result.response[:200]}...")
                    return result.response # Return the response content

                # --- Exception Handling during Setup ---
                except ImportError as ie:
                        self.logger.error(f"Import error during GraphRAG setup: {str(ie)}. Make sure all graphrag dependencies are installed.", exc_info=True)
                        raise # Re-raise import error as it's fundamental
                except FileNotFoundError as fnf:
                    self.logger.error(f"Parquet file not found during GraphRAG setup: {str(fnf)}", exc_info=True)
                    raise ValueError(f"Required GraphRAG data file missing: {fnf.filename}") from fnf
                except Exception as e:
                    # Modified log message for clarity
                    self.logger.error(f"Error during GraphRAG setup OR initial search: {str(e)}", exc_info=True) # Log traceback
                    # Fall back to direct LLM if search engine setup fails
                    try:
                        self.logger.warning("Falling back to direct LLM query due to setup/search error.")
                        # Use invoke for GraphRAGChatOpenAI or compatible method
                        response = self.llm.invoke(prompt)
                        # Extract content based on expected response structure
                        llm_response_content = getattr(response, 'content', str(response))
                        self.logger.info(f"LLM Fallback Response: {llm_response_content[:200]}...")
                        return llm_response_content
                    except Exception as llm_error:
                        self.logger.error(f"Both search engine setup/search and LLM fallback failed: {str(llm_error)}", exc_info=True)
                        # Provide a more informative error message
                        return f"Error processing query. GraphRAG setup/search failed: {str(e)}. Fallback LLM failed: {str(llm_error)}".strip()

            else:
                # --- Use Cached Search Engine ---
                try:
                    self.logger.info("Using cached GraphRAG search engine.")
                    self.logger.info(f"Performing cached search with prompt: {prompt[:100]}...")
                    # +++ Add check here too for safety +++
                    if self.search_engine is None:
                        self.logger.error("!!! Critical: Cached self.search_engine is None before search call !!!")
                        raise TypeError("Cached LocalSearch engine is None")
                    # +++ End added check +++
                    result = self.search_engine.search(query=prompt) # Use 'search' method
                    self.logger.info(f"Cached search completed. Result response: {result.response[:200]}...")
                    return result.response # Return the response content

                except Exception as e:
                    self.logger.error(f"Error during cached GraphRAG search execution: {str(e)}", exc_info=True)
                    # Fallback to direct LLM query if cached search fails
                    try:
                        self.logger.warning("Falling back to direct LLM query due to cached search error.")
                        response = self.llm.invoke(prompt)
                        llm_response_content = getattr(response, 'content', str(response))
                        self.logger.info(f"LLM Fallback Response: {llm_response_content[:200]}...")
                        return llm_response_content
                    except Exception as fallback_error:
                        self.logger.error(f"Fallback LLM also failed after cached search error: {str(fallback_error)}", exc_info=True)
                        return f"Error processing query. Cached GraphRAG search failed: {str(e)}. Fallback LLM failed: {str(fallback_error)}".strip()

        except Exception as outer_e:
            # Catch any unexpected errors not handled within the setup/cached blocks
            self.logger.error(f"Unexpected error in GraphRAG query method: {str(outer_e)}", exc_info=True)
            return f"Unexpected error processing GraphRAG query: {str(outer_e)}".strip()

class InferenceFactory:
    """Factory for creating appropriate inference objects"""
    @staticmethod
    def create_inference(config: SystemConfig, logger: CustomLogger) -> InferenceBase:
        inference_types = {
            'llm': LLMInference,
            'llm_rag': RAGInference,
            'llm_rag_graph': GraphRAGInference
        }

        inference_class = inference_types.get(config.inference_type)
        if not inference_class:
            raise ValueError(f"Unsupported inference type: {config.inference_type}")

        return inference_class(config, logger)

### M7 - Design Structure Matrix ###

In [7]:
class DesignStructureMatrix:
    """Manages the creation and analysis of Design Structure Matrix"""
    def __init__(self, config: SystemConfig, logger: CustomLogger):
        self.config = config
        self.logger = logger

        # Create appropriate inference based on type
        if config.inference_type =="llm":
            self.inference = InferenceFactory.create_inference(config, self.logger)
            self.logger.info(f"Initialized LLM inference for system: {self.config.concept_name}")
        elif config.inference_type == "llm_rag":
            self.inference = InferenceFactory.create_inference(config, self.logger)
            self.logger.info(f"Initialized RAG inference for system: {self.config.concept_name}")
        elif config.inference_type == "llm_rag_graph":
            self.inference = InferenceFactory.create_inference(config, self.logger)
            self.logger.info(f"Initialized GraphRAG inference for system: {self.config.concept_name}")

        self.original_components: List[str] = []
        self.predicted_components: List[str] = []
        self.components: List[str] = []
        self.matrix: Optional[pd.DataFrame] = None
        self.timestamp = time.strftime("%Y%m%d-%H%M%S")
        self.experiment_name = os.path.splitext(os.path.basename(self.config.config_path))[0]

        self.logger.info(f"Initialized DSM for system: {self.config.concept_name}")
        self.logger.info(f"Inference model: {self.config.selected_model}")

    def _query(self, prompt: str) -> str:
        """Query the relationship between two components with document validation"""
        if self.config.inference_type == "llm_rag":
            if not self._validate_documents():
                raise ValueError("Required documents are missing or inaccessible")
        
        # Get the number of components
        if config.case == 'i':
            n_components = len(self.original_components)
            self.logger.info(f'Number of components: {n_components}')
            config.predicted_components = self.original_components
        elif config.case == 'ii':
            config.predicted_components = self.predicted_components
            n_components = len(config.predicted_components)
            self.logger.info(f'Predicted components for query: {config.predicted_components}')      
        
        try:
            prompt = self.update_dsm_prompt(config)
            self.logger.info(f"Query: {prompt}")
            response = self.inference.query(prompt)
            self.logger.info(f"Response: {response}")
            # Extract matrix from response
            try:
                response = self.response_filter(response)
                
                # Look for the final response indicator
                final_response_indicator = "final response = ["
                matrix_start = response.find(final_response_indicator)
                
                if matrix_start != -1:
                    # Adjust start position to the beginning of the actual list
                    matrix_start = matrix_start + len(final_response_indicator) - 1  # -1 to keep the '['
                    matrix_end = response.rfind(']') + 1
                    matrix_str = response[matrix_start:matrix_end]
                else:
                    # Fallback to looking for any list if final response indicator not found
                    matrix_start = response.find('[')
                    matrix_end = response.rfind(']') + 1
                    if matrix_start == -1 or matrix_end == -1:
                        raise ValueError("No valid matrix found in response")
                    matrix_str = response[matrix_start:matrix_end]

                # Validate and parse the matrix
                matrix = eval(matrix_str)
                self.logger.info(f"Matrix: {matrix}")
                
                # Verify it is a valid nxn matrix of numbers where n is the number of components
                if (isinstance(matrix, list) 
                    and len(matrix) == n_components 
                    and all(isinstance(row, list) and len(row) == n_components for row in matrix)):
                    return matrix
                else:
                    raise ValueError(f"Invalid matrix format - expected {n_components}x{n_components} matrix")
                    
            except Exception as e:
                self.logger.error(f"Error parsing matrix: {str(e)}")
                # Return a default "unknown" matrix of correct size
                return [[2 for _ in range(n_components)] for _ in range(n_components)]
                
        except Exception as e:
            self.logger.error(f"Error in query: {str(e)}")
            return [[2 for _ in range(n_components)] for _ in range(n_components)]

    def update_dsm_prompt(self, config: SystemConfig):
        """Create the DSM prompt with current config components"""
        return f"""
        Please identify system-level {config.relationship_type} relationships between the components of {config.concept_name}, as listed below, and represent them in a Design Structure Matrix (DSM) format.  \
        
        The relationships should be articulated in plain English, capturing the essence of their interactions. \
        
        Here is the list of subsystems/components to analyze: {config.predicted_components} \
        
        The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
        
        Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = {len(config.predicted_components)} for this example), with rows and columns labeled accordingly. \
        
        Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \
        
        If you don't know the answer, just write 2 for the specific cell that you are unsure of. A worst case scenario of the **as a list of lists** shown below (diagonal elements are 1 and unsure ones are 2). Don't make up an answer. \
        
        final response = [[1, 2, 2, 2, 2, 2, 2],[2, 1, 2, 2, 2, 2, 2],[2, 2, 1, 2, 2, 2, 2],[2, 2, 2, 1, 2, 2, 2],[2, 2, 2, 2, 1, 2, 2],[2, 2, 2, 2, 2, 1, 2],[2, 2, 2, 2, 2, 2, 1]] for 7 components \
        
        final response = [[1, 2, 2],[2, 1, 2],[2, 2, 1]] for 3 components \
        
        IMPORTANT: Your response must **ONLY** contain a valid **as a list of lists** in the following example format 7x7 matrix (NO LATEX, NO CODE, NO DESCRIPTION, OR OTHER ENCODINGS, JUST THE LIST OF LISTS). \
        
        The below list of lists is just examples format of a response. **Replace** the values with your **actual analysis** while providing your response with parameter of 'final response'.\
        
        final response = [[1, 0, 0, 0, 0, 0, 0],[0, 1, 0, 0, 0, 0, 0],[0, 0, 1, 0, 0, 0, 0],   [0, 0, 0, 1, 0, 0, 0],[0, 0, 0, 0, 1, 0, 0],[0, 0, 0, 0, 0, 1, 0],[0, 0, 0, 0, 0, 0, 1]] for 7 components \
        
        final response = [[1, 0, 0],[0, 1, 0],[0, 0, 1]] for 3 components \
        
        IMPORTANT: THE FORMAT OF THE OUTPUT MUST BE AS LIST OF LISTS ONLY SIMILAR TO THE ONE ABOVE (NO LATEX, NO CODE, NO DESCRIPTION, OR OTHER ENCODINGS, JUST THE LIST OF LISTS). \
        """

    def _validate_documents(self) -> bool:
        """Validate that required documents are available"""
        if not self.config.reference_files:
            self.logger.error("No reference files configured")
            return False

        for r_type, files in self.config.reference_files.items():
            for file in files:
                if not os.path.exists(file['path']):
                    self.logger.error(f"Missing document: {file['path']}")
                    return False
        return True

    def identify_components(self, max_retries: int = 3, current_try: int = 1) -> List[str]:
        """Identify system components using selected inference approach"""
        if config.inference_type != "llm" and not self._validate_documents():
            raise ValueError("Required documents are missing or inaccessible")
        
        try:
            # Prompt 1
            # prompt = (
            #     f"Specify the key {self.config.relationship_type} components forming "
            #     f" a {self.config.system_name} in the {self.config.application_domain}"
            #     f" domain. Return your response as a textual list and a Python list."
            #     f" If you cannot determine the answer, respond with I don't know."
            # )
            # # Prompt 2
            # prompt = (
            #     f"Identify the system-level components that form the core architecture of a {self.config.system_name} in the {self.config.application_domain}." \
            #     f"Consider only major subsystems that are essential for the system's primary functions, avoiding detailed parts or minor elements. " \
            #     f"Format your response as: A numbered list of subsystems with their core functions; A Python list containing just the subsystem names as strings. " \
            #     f"Target a conceptual design level with approximately 8-15 major subsystems. If you cannot determine the essential subsystems, respond with 'I don't know'."
            # )
            # Prompt
            prompt = f"""
            Please identify the major components of {self.config.concept_name} in the {self.config.application_domain} based on {self.config.relationship_type} relationships. \
            
            The number of components must be {len(self.original_components)} and must be a list of strings. \
            
            Please do not identify similar components that will work together under subsystem level since our interest is to identify the major components that form the core architecture of the system. \
            
            Here is an example of the components for made-up system: \
            
            final response = [ "component 1", "component 2", "component 3", "component 4", "component 5" ] \
            
            IMPORTANT: THE FORMAT OF THE OUTPUT MUST BE AS A LIST OF STRINGS ONLY SIMILAR TO THE ONE ABOVE (NO LATEX, NO CODE, NO DESCRIPTION, OR OTHER ENCODINGS).  \
            """

            self.logger.info(f"Prompt (attempt {current_try}/{max_retries}): {prompt}")
            raw_response = self.inference.query(prompt)

            # Extract Python list from response with better error handling
            try:
                raw_response = self.response_filter(raw_response)
                
                # Look for the final response indicator
                final_response_indicator = "final response = ["
                matrix_start = raw_response.find(final_response_indicator)
                
                if matrix_start != -1:
                    # Adjust start position to the beginning of the actual list
                    matrix_start = matrix_start + len(final_response_indicator) - 1  # -1 to keep the '['
                    matrix_end = raw_response.rfind(']') + 1
                    matrix_str = raw_response[matrix_start:matrix_end]
                else:
                    # Fallback to looking for any list if final response indicator not found
                    matrix_start = raw_response.find('[')
                    matrix_end = raw_response.rfind(']') + 1
                    if matrix_start == -1 or matrix_end == -1:
                        raise ValueError("No valid matrix found in response")
                    matrix_str = raw_response[matrix_start:matrix_end]

                # Validate and parse the matrix
                components = eval(matrix_str)
                self.logger.info(f"Matrix: {components}")

                # list_start = raw_response.index('[')
                # list_end = raw_response.index(']') + 1
                # components = eval(raw_response[list_start:list_end])

                # Validate components
                if not isinstance(components, list) or not all(isinstance(x, str) for x in components):
                    raise ValueError("Invalid component list format")

                self.components = components
                self.logger.info(f"Components: {self.components}")

                # Validate response with a secondary check
                if self.config.check_enable_validation:
                    if self.validate_component_response(prompt, raw_response):
                        self.logger.info(f"Component list validated successfully on attempt {current_try}")
                        return self.components
                    else:
                        raise ValueError(f"Failed to validate components after {max_retries} attempts")
                else:
                    if current_try < max_retries:
                        self.logger.warning(f"Retrying component identification (attempt {current_try + 1}/{max_retries})")
                        return self.identify_components(max_retries, current_try + 1)
                    else:
                        self.logger.error(f"Could not extract valid component list after {max_retries} attempts")
                        raise ValueError(f"Could not extract valid component list after {max_retries} attempts")
                
            except ValueError as e:
                self.logger.error(f"Error parsing component list: {str(e)}")
                self.logger.error(f"Erroneous raw response: {raw_response}")
                if current_try < max_retries:
                    self.logger.warning(f"Retrying component identification (attempt {current_try + 1}/{max_retries})")
                    return self.identify_components(max_retries, current_try + 1)
                raise ValueError(f"Could not extract valid component list after {max_retries} attempts")
            
        except Exception as e:
            self.logger.error(f"Error identifying components: {str(e)}")
            if current_try < max_retries:
                self.logger.warning(f"Retrying due to error (attempt {current_try + 1}/{max_retries})")
                return self.identify_components(max_retries, current_try + 1)
            raise
    
    def validate_component_response(self, original_prompt: str, raw_response: str) -> bool:
        """Validate if the LLM's component identification response matches requirements"""
        validation_prompt = f"""
        You are a validator. Review this response based on the requirements of the original prompt: \
        
        Original prompt: {original_prompt}
        Response: {raw_response}
        
        Requirements:
        1- Response must follow the format in the original prompt.
        2- Response must have the same number of components as the original prompt.
        
        Answer only with 'Valid' or 'Invalid'.
        """

        try:
            validation_response = self.inference.query(validation_prompt)
            self.logger.info(f"Validation response: {validation_response}")
            validation_response = self.response_filter(validation_response)
            self.logger.info(f"Filtered validation response: {validation_response}")
            
            is_valid = validation_response.strip().upper().startswith('VALID')
            if not is_valid:
                self.logger.warning("Validation failed: {validation_response}")
            return is_valid
        except Exception as e:
            self.logger.error(f"Error in validation: {str(e)}")
            return False
    
    def response_filter(self, response: str) -> str:
        """Filter the response to remove any unwanted content"""
        # First try to find content after </think> if it exists
        think_end = response.find("</think>")
        if think_end != -1:
            response = response[think_end + len("</think>"):]
        
        # Remove any line containing triple backticks
        response = '\n'.join(line for line in response.splitlines() if '```' not in line)

        return response

    def analyze_component_relationships(self) -> pd.DataFrame:
        """Create and fill the DSM with component relationships using parallel processing"""
        if not self.components:
            raise ValueError("Components must be identified before analyzing relationships")

        # Create component pairs excluding self-relationships
        component_pairs = [
            (source, target)
            for source in self.components
            for target in self.components
            if source != target
        ]

        def process_pair(pair: Tuple[str, str]) -> Tuple[str, str, int]:
            """Process a single component pair"""
            source, target = pair
            relationship = self._query_relationship(source, target)
            return (source, target, relationship)

        # Process relationships in parallel
        with ThreadPoolExecutor(max_workers=5) as executor:
            results = list(executor.map(process_pair, component_pairs))

        # Convert results to DataFrame
        relationship_df = pd.DataFrame(
            results,
            columns=['source', 'target', 'relationship']
        )

        # Create matrix using pivot
        matrix = pd.pivot_table(
            relationship_df,
            values='relationship',
            index='target',
            columns='source',
            fill_value=0
        )

        # Add diagonal of ones and ensure proper component ordering
        matrix = matrix.reindex(
            index=self.components,
            columns=self.components,
            fill_value=0
        )
        np.fill_diagonal(matrix.values, 1) # in DSM components are self-connected

        self.matrix = matrix

        self.logger.info("\nFinal Design Structure Matrix:")
        self.logger.info(f"\n{self.matrix}")

        return matrix
    
    def log_experiment(self, message: str) -> None:
        """Log experiment information using the logger"""
        self.logger.info(message)

    def _query_relationship(self, component1: str, component2: str) -> int:
        """Query the relationship between two components"""
        # Prompt 1
        # question = (
        #     f"Are {component1} and {component2} connected {self.config.relationship_type}-ly?"
        #      "Provide 'Yes,' 'No,' or 'I don't know. I need more information' if you're not sure or uncertain."
        #      "Please do not write more than these three answers."
        # )
        
        # Prompt 2
        question = f"""
            Do {component1} and {component2} have a {self.config.relationship_type}-based interaction or dependency that is essential for the {self.config.concept_name}'s core functionality? \
            
            Provide 'Yes,' 'No,' or 'I don't know. I need more information' if you're not sure or uncertain. \
            
            Please do not write anything other than these three answers.
        """

        response = self.inference.query(question)
        # self.logger.info(f"Identifying {self.config.relationship_type} relationship: {component1} -> {component2}") # Prompt 1
        self.logger.info(f"Identifying {self.config.relationship_type} relationship: {component1} -> {component2}") # Prompt 2
        self.logger.info(f"Answer: {response}")

        if "Yes" in response:
            return 1
        elif "No" in response:
            return 0
        return 2

    def save_results(self) -> None:
        """Save the DSM and analysis results"""
        if self.matrix is None:
            raise ValueError("Matrix has not been created yet")

        Path(self.config.output_directory).mkdir(parents=True, exist_ok=True)

        # Save matrix to CSV
        matrix_path = os.path.join(
            self.config.output_directory,
            f'{self.experiment_name}_dsm_{self.timestamp}.csv'
        )
        self.matrix.to_csv(matrix_path)

        # Save analysis summary
        summary_path = os.path.join(
            self.config.output_directory,
            f'{self.experiment_name}_analysis_summary_{self.timestamp}.json'
        )
        summary = {
            'timestamp': self.timestamp,
            'system_name': self.config.system_name,
            'inference_type': self.config.inference_type,
            'model_used': self.config.selected_model,
            'components': self.components,
            'num_components': len(self.components)
        }
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2)

### M8 - Run Multiple Experiments

In [8]:
import subprocess
from pathlib import Path
from typing import List, Dict, Optional, Union
import numpy as np
from collections import Counter
import time
import json
import os
from itertools import combinations
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from datetime import datetime, timedelta
from dataclasses import dataclass, field

@dataclass
class ModelExperiment:
    """Configuration for running multiple iterations of a single model"""
    n_runs: int
    components: list[str]

@dataclass
class PowerDrillExperiment:
    """Configuration for running multiple iterations of a single model"""
    n_runs: int = 5
    components: list[str] = field(default_factory=lambda: [
        'Bit', 'Transmission', 'Motor', 'Electrical System',
        'Battery Holder', 'Housing', 'External Environment'
    ])

@dataclass
class CubeSatExperiment:
    """Configuration for running multiple iterations of a single model"""
    n_runs: int = 5
    components: list[str] = field(default_factory=lambda: [
        'Spacecraft', 'Guidance, Navigation, and Control (GNC) Subsystem',
        'Propulsion Subsystem', 'Power Subsystem', 'Reaction Wheel', 'GNC Software'
    ])

def reset_gpus():
    """Reset GPUs using the reset scripts"""
    try:
        # Get the home directory
        home = str(Path.home())
        reset_script = f"./gpu_reset.sh" # xCraft/gpu_reset.sh

        print("\nResetting GPUs...")
        result = subprocess.run(
            ['bash', reset_script],
            capture_output=True,
            text=True
        )

        if result.returncode == 0:
            print("GPUs reset successfully")
        else:
            print(f"GPU reset failed with error: {result.stderr}")

        # Wait after reset 
        print("Waiting for 15 seconds after GPU reset...")
        time.sleep(15)

    except Exception as e:
        print(f"Error during GPU reset: {str(e)}")
        time.sleep(15) # Still wait even if reset fails

def generate_reference_combinations(base_references: Dict[str, List[Dict[str, str]]]) -> List[Dict[str, List[Dict[str, str]]]]:
    """Generate all possible combinations of reference files"""
    ref_types = list(base_references.keys())
    all_combinations = []
    
    # Generate combinations of different lengths (1 to len(ref_types))
    for r in range(1, len(ref_types) + 1):
        for combo in combinations(ref_types, r):
            ref_config = {}
            for ref_type in combo:
                ref_config[ref_type] = base_references[ref_type]
            all_combinations.append(ref_config)
    
    return all_combinations

def run_model_experiment(
    base_config: SystemConfig,
    model: str,
    experiment_config: Any, # Use specific type like PowerDrillExperiment
    logger: CustomLogger,
    prompt: str = None # DSM prompt template for case ii
) -> List[Dict]:
    """
    Run multiple iterations for a model, handling RAG, LLM, and GraphRAG
    using the original structure.
    """

    all_results = []
    timestamp_start_experiment = datetime.now()
    logger.info(f"Starting experiment for model: {model} at {timestamp_start_experiment}")
    logger.info(f"Inference type: {base_config.inference_type}")

    # --- Adjust Reference Handling Based on Inference Type ---
    if base_config.inference_type == "llm_rag_graph":
        # For GraphRAG, we run once on the full indexed graph.
        # Create a single dummy combination representing the full graph.
        ref_combinations = [{"indexed_graph": []}] # Use a specific key like "indexed_graph"
        logger.info("GraphRAG mode: Will run on the single indexed graph.")
    elif base_config.inference_type == "llm":
        # For LLM, run once with no specific references (baseline).
        ref_combinations = [{"baseline": []}]
        logger.info("LLM mode: Will run baseline.")
    else: # RAG
        # For RAG, generate combinations from provided reference files.
        if not base_config.reference_files:
            logger.error("Reference files must be provided for RAG inference type.")
            return [] # Cannot proceed
        ref_combinations = generate_reference_combinations(base_config.reference_files)
        logger.info(f"RAG mode: Generated {len(ref_combinations)} reference combinations.")

    # --- Loop through combinations (will be just one for GraphRAG/LLM) ---
    for ref_combo in ref_combinations:
        # Determine ref_types based on the keys in the current combo
        ref_types = list(ref_combo.keys())
        logger.info(f"--- Starting Combination: {ref_types} ---")

        # Initialize lists for this combination's runs
        run_matrices = []
        run_results = []
        components = [] # Store identified components for this combo

        timestamp_start_combo = datetime.now()

        # --- Inner loop for n_runs (Structure remains the same) ---
        for run in range(experiment_config.n_runs):
            logger.info(f"--- Run {run + 1}/{experiment_config.n_runs} for combo {ref_types} ---")
            try:
                # Create config for this specific run
                # For GraphRAG, ref_combo will be {"indexed_graph": []}, which is fine.
                # For RAG, it uses the actual reference file subset.
                run_config = SystemConfig(
                    **{**base_config.__dict__, # Start with base config
                       'selected_model': model,
                       'reference_files': ref_combo, # Pass current combo
                       # Output directory might need adjustment if saving per run later
                       # 'output_directory': f"{base_config.output_directory}/{model}/{'_'.join(ref_types)}/run_{run+1}"
                      }
                )
                
                # Update GraphRAG input directory for the specific model if using GraphRAG
                if run_config.inference_type == "llm_rag_graph":
                    # Format the model name for directory path
                    model_dir_name = model.replace(':', '_').replace('/', '_')
                    # Create appropriate path based on system name and model
                    system_prefix = "PS" if run_config.system_name == "PowerDrill" else "CS"  # PS for PowerDrill, CS for CubeSat
                    # Get embedding suffix from the embedding model
                    embedding_suffix = "te3s" if "text-embedding-3" in run_config.embedding_model else "ada002"
                    # Set the correct GraphRAG input directory
                    run_config.graphrag_input_dir = f"../data/ragtest/{system_prefix}/auto_{model_dir_name}-{embedding_suffix}"

                # --- Instantiation inside the loop (as per original code) ---
                # InferenceFactory will create GraphRAGInference if type matches.
                # GraphRAGInference.__init__ handles the index check internally.
                dsm = DesignStructureMatrix(run_config, logger)
                # logger.info(f"Experiment config components: {experiment_config.components}")
                dsm.original_components = experiment_config.components
                # logger.info(f"DSM original components: {dsm.original_components}")

                # --- Case 'i' / 'ii' Logic (Remains the same) ---
                if run_config.case == "i":
                    # Directly query for matrix
                    # Ensure 'prompt' is suitable for case 'i' if this path is used
                    components = experiment_config.components # Separate than dsm.original_components for experiment config.
                    logger.info("Running Case 'i': Direct DSM query.")
                    response = dsm._query(prompt)
                elif run_config.case == "ii":
                    # Identify components then query matrix
                    if run == 0:
                        logger.info("Running Case 'ii': Identifying components (Run 0)...")
                        components = dsm.identify_components()
                        logger.info(f"Components identified: {components}")
                        if not components:
                             logger.error("Failed to identify components in run 0. Aborting for this combination.")
                             # Break out of the n_runs loop for this combo
                             run_matrices = [] # Clear matrices to prevent averaging bad data
                             run_results = []
                             break
                    else:
                         # Reuse components from run 0
                         dsm.components = components # Set identified components on the DSM instance
                         logger.info(f"Reusing components from run 0: {components}")

                    # Prepare config and prompt for DSM query
                    run_config.predicted_components = components # Update config for _query method
                    dsm.predicted_components = components
                    # Format the DSM prompt template with current components/config
                    formatted_dsm_prompt = prompt.format(
                        relationship_type=run_config.relationship_type,
                        concept_name=run_config.concept_name,
                        predicted_components=run_config.predicted_components,
                        # component_list_str=str(components), # Convert list to string for prompt
                        n_components=len(components)
                    )
                    logger.info("Querying for DSM matrix...")
                    response = dsm._query(formatted_dsm_prompt) # _query handles parsing

                # --- Parsing Logic (Remains the same) ---
                # This block needs to correctly parse the string output from
                # LLM, RAG, or GraphRAG query methods.
                matrix = None
                try:
                    if isinstance(response, str):
                        # Add robust parsing here if needed, potentially using dsm.response_filter
                        # Example: clean_response = dsm.response_filter(response)
                        # Find the list part:
                        list_start = response.find('[')
                        list_end = response.rfind(']') + 1
                        if list_start != -1 and list_end != 0:
                             matrix_str = response[list_start:list_end]
                             logger.debug(f"Attempting to eval matrix string: {matrix_str[:100]}...")
                             matrix = eval(matrix_str)
                        else:
                             raise ValueError("Could not find list markers '[]' in response string.")
                    elif isinstance(response, list):
                        logger.info("Query returned a list directly.")
                        matrix = response
                    else:
                        raise TypeError(f"Unexpected response type: {type(response)}")

                    # --- Add validation for matrix size/content ---
                    n_comps_expected = len(components) if run_config.case == "ii" else len(dsm.original_components)
                    if not (isinstance(matrix, list) and
                            len(matrix) == n_comps_expected and
                            all(isinstance(row, list) and len(row) == n_comps_expected for row in matrix)):
                        raise ValueError(f"Parsed matrix has incorrect format/size. Expected {n_comps_expected}x{n_comps_expected}.")

                    run_matrices.append(matrix)
                    logger.info(f"Matrix obtained and validated for run {run + 1}:\n{np.array(matrix)}")

                    # Store result for this run
                    run_results.append({
                        'run_number': run + 1, # Use 1-based index
                        'matrix': matrix,
                        'timestamp': datetime.now().isoformat()
                    })

                except Exception as e:
                    logger.error(f"Error parsing or validating matrix for run {run + 1}: {str(e)}", exc_info=True)
                    logger.error(f"Failed response content: {response}")
                    # Decide how to handle run failure (e.g., skip run, use default matrix?)
                    # Skipping run for now:
                    continue # Skip to next run

                # Delay between runs
                sleep_duration = 10
                logger.debug(f"Sleeping for {sleep_duration} seconds...")
                time.sleep(sleep_duration)

            except Exception as e:
                logger.error(f"Error within run {run + 1} for combo {ref_types}: {str(e)}", exc_info=True)
                # Continue to next run even if one fails catastrophically
                continue

        # --- Aggregation (Remains the same position: after n_runs, inside ref_combo loop) ---
        timestamp_end_combo = datetime.now()
        logger.info(f"Finished {experiment_config.n_runs} runs for combo {ref_types}. Duration: {timestamp_end_combo - timestamp_start_combo}")

        if run_matrices: # Only if we have matrices to average
            logger.info(f"Aggregating results for combo: {ref_types}")
            matrices_array = np.array(run_matrices)
            # Handle '2' (unknown) during averaging
            matrices_array_float = matrices_array.astype(float)
            matrices_array_float[matrices_array_float == 2] = np.nan # Treat 2 as NaN
            average_matrix_float = np.nanmean(matrices_array_float, axis=0)
            # Convert back: NaN -> 2, otherwise threshold >= 0.5 -> 1, else 0
            average_matrix = np.where(np.isnan(average_matrix_float), 2, (average_matrix_float >= 0.5).astype(int))
            logger.info(f"Computed average matrix for combo {ref_types}:\n{average_matrix}")

            # Store results for this combination
            combo_results = {
                'model': model,
                'system': config.system_name,
                'reference_types': config.reference_types, # Will be ["indexed_graph"] for GraphRAG
                'average_matrix': average_matrix,
                'individual_runs': run_results,
                'components': components # Store the identified components for this combo
            }
            all_results.append(combo_results)
        else:
            logger.warning(f"No successful runs completed for model {model} with combo {ref_types}. No average matrix generated.")

    # --- End of loops ---
    timestamp_end_experiment = datetime.now()
    logger.info(f"Finished experiment for model: {model}. Total duration: {timestamp_end_experiment - timestamp_start_experiment}")
    return all_results

def save_model_results(results: List[Dict], config: SystemConfig, base_path: str, logger: CustomLogger, timestamp: str):
    """Save model experiment results for all reference combinations"""
    # Check if the entire results list is empty
    if not results:
        logger.warning(f"No valid results generated for model associated with config {config.selected_model} and timestamp {timestamp}. Skipping saving.")
        return

    for result in results: # each result is for a reference type i.e. R1, R2,..., R1-R2-R3, etc. collectively as results

        # Check for empty/invalid individual result dictionary before proceeding
        is_invalid = False
        if not result:
            is_invalid = True
            logger.warning("Encountered an empty result dictionary. Skipping.")
        elif 'average_matrix' not in result or 'components' not in result:
            is_invalid = True
            logger.warning(f"Result dictionary missing 'average_matrix' or 'components' key. Skipping. Keys found: {list(result.keys())}")
        elif not result['components']: # Check if components list is empty
             is_invalid = True
             logger.warning(f"Result dictionary has an empty 'components' list. Skipping.")
        # Check if average_matrix is a numpy array and has size > 0
        elif not isinstance(result['average_matrix'], np.ndarray) or result['average_matrix'].size == 0:
             is_invalid = True
             logger.warning(f"Result dictionary has an empty or invalid 'average_matrix'. Type: {type(result['average_matrix'])}, Size: {getattr(result['average_matrix'], 'size', 'N/A')}. Skipping.")

        if is_invalid:
            # Try to log context if possible
            model_name_from_result = result.get('model', config.selected_model) if result else config.selected_model
            ref_types_from_result = result.get('reference_types', 'N/A') if result else 'N/A'
            logger.warning(f"--> Skipping saving invalid result entry for model '{model_name_from_result}', ref_types '{ref_types_from_result}'")
            continue # Skip to the next result in the list

        # --- Original saving logic starts here if the result is valid ---
        logger.info("\n" + "="*50)  # Clear section separator
        logger.info(f"Average DSM Matrix for model: {result['model']}")
        logger.info(f"End of log reference type(s): {result['reference_types']}")

        model_name = result['model'].replace(':', '_').replace('/', '_') # Make model name path-safe
        ref_types_str = '_'.join(result['reference_types']) # Make ref_types path-safe

        # Create specific directory for this combination
        # Ensure the inference type used in the path matches the current config
        combo_path = os.path.join(base_path, f"{config.inference_type}_refs_{ref_types_str}")
        try:
            os.makedirs(combo_path, exist_ok=True)
            logger.info(f"Saving results to: {combo_path}")
        except OSError as e:
            logger.error(f"Failed to create directory {combo_path}: {e}. Skipping save for this result.")
            continue


        # Save average matrix as CSV
        # Wrap DataFrame creation in a try-except block just in case, although checks above should prevent it
        try:
            average_df = pd.DataFrame(
                result['average_matrix'],
                index=result['components'],
                columns=result['components']
            )
        except ValueError as e:
            logger.error(f"Error creating DataFrame despite previous checks: {e}. Matrix shape: {result['average_matrix'].shape}, Components: {len(result['components'])}. Skipping save for this result.")
            continue
        except Exception as e: # Catch any other unexpected error during DataFrame creation
            logger.error(f"Unexpected error creating DataFrame: {e}. Skipping save for this result.")
            continue


        # Convert DataFrame to string with proper formatting
        matrix_str = "\nAverage Matrix:\n" + average_df.to_string()
        logger.info(matrix_str)
        logger.info(f"\nComponents: {result['components']}")
        logger.info("="*50 + "\n")

        csv_path = os.path.join(combo_path, f"average_matrix_{timestamp}.csv") # Use os.path.join
        try:
            average_df.to_csv(csv_path)
            logger.info(f"Saved CSV to: {csv_path}")
        except Exception as e:
            logger.error(f"Failed to save CSV to {csv_path}: {e}")
            # Decide if you want to continue saving JSON/PNG or skip the rest for this result

        # Save detailed results
        json_path = os.path.join(combo_path, f"detailed_results_{timestamp}.json") # Use os.path.join
        try:
            # Ensure average_matrix is converted to list for JSON serialization
            serializable_matrix = result['average_matrix'].tolist() if isinstance(result['average_matrix'], np.ndarray) else result['average_matrix']

            serializable_results = {
                'model': result['model'],
                'system': result['system'],
                'reference_types': result['reference_types'],
                'average_matrix': serializable_matrix, # Use the converted matrix
                'individual_runs': [{
                    'run_number': r['run_number'],
                    # Ensure individual matrices are also serializable if they are numpy arrays
                    'matrix': r['matrix'].tolist() if isinstance(r.get('matrix'), np.ndarray) else r.get('matrix'),
                    'timestamp': r.get('timestamp', 'N/A') # Use .get for safety
                } for r in result.get('individual_runs', [])], # Use .get for safety
                'components': result['components']
            }
            with open(json_path, 'w') as f:
                json.dump(serializable_results, f, indent=2)
            logger.info(f"Saved JSON to: {json_path}")
        except TypeError as e:
             logger.error(f"Failed to serialize results to JSON at {json_path}: {e}")
        except Exception as e:
            logger.error(f"Failed to save JSON to {json_path}: {e}")

        # Create and save visualization
        try:
            plt.figure(figsize=(10, 8))
            sns.heatmap(
                average_df, # Use the DataFrame created earlier
                annot=True,
                cmap='YlOrRd',
                fmt='.2f', # Use float formatting for average
                square=True
            )
            plt.title(f'Average DSM Matrix - {model_name}\nReferences: {ref_types_str}') # Use path-safe names
            plt.tight_layout()
            png_path = os.path.join(combo_path, f"average_matrix_{timestamp}.png") # Use os.path.join
            plt.savefig(png_path)
            plt.close() # Close the plot to free memory
            logger.info(f"Saved visualization to: {png_path}")
        except Exception as e:
            logger.error(f"Failed to create or save visualization: {e}")
            plt.close() # Ensure plot is closed even if saving fails


## ====== Experiments ======

#### E: Experiment; M: Multiple Experiments; P: Power Screwdriver; C: CubeSat; L: Baseline LLM; R: Baseline RAG; G: Baseline GraphRAG

#### Example: iMPL -> i Identification of the existent part relationship, multiple experiment, Power Screwdriver, Baseline LLM

### i: Power Screwdriver


#### E-iMPL - Running Multiple Experiments with Baseline LLM

In [ ]:
# Example usage:
if __name__ == "__main__":
    # Create the base config
    # Create the base config
    config = SystemConfig(
        system_name="PowerDrill",
        concept_name="PowerDrill",
        application_domain="engineering",
        relationship_type="proximity",
        api_keys={'openai_api_key': os.getenv('OPENAI_API_KEY')},
        selected_model="gpt-4-turbo-preview",
        ollama_api_key="ollama",
        ollama_api_base="http://localhost:11434/v1",
        ollama_emb_base="http://localhost:11434/api",
        embedding_model= "nomic-embed-text", #"text-embedding-3-small",
        inference_type="llm",
        reference_files={},
        output_directory="../output",
        vectorstore_directory="",
        check_enable_validation=False,
        case="i"
    )
    
    # Your DSM prompt
    # _prompt = """
    # Please identify system-level proximity relationships (in contact) between the components of Power Screwdriver, as listed below, and represent them in a Design Structure Matrix (DSM) format.  \

    # The relationships should be articulated in plain English, capturing the essence of their interactions. \

    # Here is the list of subsystems/components to analyze: ['Bit', 'Transmission', 'Motor', 'Electrical System', 'Battery Holder', 'Housing', 'External Environment'] \

    # The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
        
    # Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = 7 for this example), with rows and columns labeled accordingly. \

    # Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \

    # If you don't know the answer, just write 2 for the specific cell that you are unsure of. Don't make up an answer. \

    # Provide your output in **Python list format** in the same order as the components in the list above. \
    # """
    
    # Define the prompt
    _prompt = f"""
    Please identify system-level {config.relationship_type} relationships between the components of {config.concept_name}, as listed below, and represent them in a Design Structure Matrix (DSM) format.  \

    The relationships should be articulated in plain English, capturing the essence of their interactions. \

    Here is the list of subsystems/components to analyze: {config.predicted_components} \

    The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
        
    Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = {len(config.predicted_components)} for this example), with rows and columns labeled accordingly. \

    Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \

    If you don't know the answer, just write 2 for the specific cell that you are unsure of. A worst case scenario of the **as a list of lists** shown below (diagonal elements are 1 and rest are 2). Don't make up an answer. \
    
    This is an example of the output format for 7 components (as 7x7 matrix)-based on our system-of-interest the size of the matrix can change. 
    
    final response = [
        [1, 2, 2, 2, 2, 2, 2],
        [2, 1, 2, 2, 2, 2, 2],
        [2, 2, 1, 2, 2, 2, 2],
        [2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 2, 1, 2, 2],
        [2, 2, 2, 2, 2, 1, 2],
        [2, 2, 2, 2, 2, 2, 1]
    ]

    IMPORTANT: Your response must **ONLY** contain a valid **as a list of lists** in the following example format 7x7 matrix. \
    
    The below list of lists is just an example format of your response. **Replace** the values with your **actual analysis** while providing your response with parameter of 'final response'.\
    
    This is an example of the output format for 7 components (as 7x7 matrix)-based on our system-of-interest the size of the matrix can change. \
    
    final response = [
        [1, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 1]
    ]

    IMPORTANT: THE FORMAT OF THE OUTPUT MUST BE AS LIST OF LISTS ONLY SIMILAR TO THE ONE ABOVE (NO LATEX, CODE, OR OTHER ENCODINGS).
    """

    # List of models to test
    models = [
        "gpt-4-turbo-preview", #43.07-42.20=
        # "gpt-4o-2024-11-20", #42.20-42.04=
        # "ollama:mixtral:8x22b_6k",
        # "ollama:llama3.3:70b-8k",
        # "ollama:deepseek-r1:14b-5k"
    ]

    experiment_config = PowerDrillExperiment(n_runs=5)
    
    # Run experiments for each model sequentially
    for model in models:
        # At the start
        timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_start = datetime.strptime(timestamp_str, "%Y%m%d-%H%M%S")
        dsm_logger = CustomLogger(config.inference_type,"MultiModelExp",model,timestamp_str)
        results = run_model_experiment(config, model, experiment_config, dsm_logger, _prompt)
        # Save results for this model with timestamp
        output_path = f"../output/experiment_{config.inference_type}_{model}_{timestamp_str}"
        save_model_results(results, config, output_path, logger=dsm_logger, timestamp=timestamp_str)
        timestamp_final_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_end = datetime.strptime(timestamp_final_str, "%Y%m%d-%H%M%S")
        dsm_logger.info(f"Duration of experiment for {model} in {config.inference_type} inference type: {timestamp_end-timestamp_start}")
        dsm_logger.cleanup() # Force flush all handlers
        time.sleep(5) # Give time for the logger to close
        reset_gpus() # Reset GPUs between models
    
    

#### E-iMPLR - Running Multiple Experiments with Baseline RAG

In [ ]:
# Example usage:
if __name__ == "__main__":
    # Create the base config
    config = SystemConfig(
        system_name="PowerDrill",
        concept_name="PowerDrill",
        application_domain="engineering",
        relationship_type="proximity",
        api_keys={'openai_api_key': os.getenv('OPENAI_API_KEY')},
        selected_model="gpt-4-turbo-preview",
        ollama_api_key="ollama",
        ollama_api_base="http://localhost:11434/v1",
        ollama_emb_base="http://localhost:11434/api",
        embedding_model= "text-embedding-3-small",#"nomic-embed-text", #"text-embedding-3-small",
        inference_type="llm_rag",
        reference_files={  # This dictionary is utilized to make sure we focus on the right context
            'R1': [{
                'path': '../data/use_cases_large/PowerDrill/reference_pdfs/R1_combined.pdf'
            }],
            # 'R2': [{
            #     'path': '../data/use_cases_large/PowerDrill/reference_pdfs/[2012 Eppinger] R2-Design structure matrix methods and applications.pdf'
            # }],
            # 'R3': [{
            #     'path': '../data/use_cases_large/PowerDrill/reference_pdfs/R3_combined.pdf'
            # }]
        }, # This will be populated later
        output_directory="../output",
        vectorstore_directory="../data/vectorstore",
        check_enable_validation=False,
        case="i"
    )   
    
    # Your DSM prompt
    # _prompt = """
    # Please identify system-level proximity relationships (in contact) between the components of Power Screwdriver, as listed below, and represent them in a Design Structure Matrix (DSM) format.  \

    # The relationships should be articulated in plain English, capturing the essence of their interactions. \

    # Here is the list of subsystems/components to analyze: ['Bit', 'Transmission', 'Motor', 'Electrical System', 'Battery Holder', 'Housing', 'External Environment'] \

    # The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
        
    # Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = 7 for this example), with rows and columns labeled accordingly. \

    # Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \

    # If you don't know the answer, just write 2 for the specific cell that you are unsure of. Don't make up an answer. \

    # Provide your output in **Python list format** in the same order as the components in the list above. \
    # """
    
    # Define the prompt
    _prompt = f"""
    Please identify system-level {config.relationship_type} relationships between the components of {config.concept_name}, as listed below, and represent them in a Design Structure Matrix (DSM) format.  \

    The relationships should be articulated in plain English, capturing the essence of their interactions. \

    Here is the list of subsystems/components to analyze: {config.predicted_components} \

    The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
        
    Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = {len(config.predicted_components)} for this example), with rows and columns labeled accordingly. \

    Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \

    If you don't know the answer, just write 2 for the specific cell that you are unsure of. A worst case scenario of the **as a list of lists** shown below (diagonal elements are 1 and rest are 2). Don't make up an answer. \
    
    This is an example of the output format for 7 components (as 7x7 matrix)-based on our system-of-interest the size of the matrix can change. 
    
    final response = [
        [1, 2, 2, 2, 2, 2, 2],
        [2, 1, 2, 2, 2, 2, 2],
        [2, 2, 1, 2, 2, 2, 2],
        [2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 2, 1, 2, 2],
        [2, 2, 2, 2, 2, 1, 2],
        [2, 2, 2, 2, 2, 2, 1]
    ]

    IMPORTANT: Your response must **ONLY** contain a valid **as a list of lists** in the following example format 7x7 matrix. \
    
    The below list of lists is just an example format of your response. **Replace** the values with your **actual analysis** while providing your response with parameter of 'final response'.\
    
    This is an example of the output format for 7 components (as 7x7 matrix)-based on our system-of-interest the size of the matrix can change. \
    
    final response = [
        [1, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 1]
    ]

    IMPORTANT: THE FORMAT OF THE OUTPUT MUST BE AS LIST OF LISTS ONLY SIMILAR TO THE ONE ABOVE (NO LATEX, CODE, OR OTHER ENCODINGS).
    """

    # List of models to test
    models = [
        "gpt-4-turbo-preview", #43.07-42.20=
        # "gpt-4o-2024-11-20", #42.20-42.04=
        # "ollama:mixtral:8x22b_6k",
        # "ollama:llama3.3:70b-8k",
        # "ollama:deepseek-r1:14b-5k"
    ]

    experiment_config = PowerDrillExperiment(n_runs=5)
    
    # Run experiments for each model sequentially
    for model in models:
        # At the start
        timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_start = datetime.strptime(timestamp_str, "%Y%m%d-%H%M%S")
        dsm_logger = CustomLogger(config.inference_type,"MultiModelExp",model,timestamp_str)
        results = run_model_experiment(config, model, experiment_config, dsm_logger, _prompt)
        # Save results for this model with timestamp
        output_path = f"../output/experiment_{config.inference_type}_{model}_{timestamp_str}"
        save_model_results(results, config, output_path, logger=dsm_logger, timestamp=timestamp_str)
        timestamp_final_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_end = datetime.strptime(timestamp_final_str, "%Y%m%d-%H%M%S")
        dsm_logger.info(f"Duration of experiment for {model} in {config.inference_type} inference type: {timestamp_end-timestamp_start}")
        dsm_logger.cleanup() # Force flush all handlers
        time.sleep(5) # Give time for the logger to close
        reset_gpus() # Reset GPUs between models
    

#### E-iMPLR - Running Multiple Experiments with Baseline GraphRAG

We can use CLI directly to find the DSM. Please make sure that you initialized, autotuned, indexed.
Please change `/path/to/model` with the target model that you initialized, autotuned, and indexed.

In [ ]:
! ./graph_rag_runner.sh query --method local \
                              --query-file "./ragtest/path/to/model/prompts/prompt_power_screwdriver_i.txt" \
                              --root ./ --gpus 0,1,2,3,4,5

### i: CubeSat

#### E-iMCL - Running Multiple Models with Baseline LLM

In [ ]:
# Example usage:
if __name__ == "__main__":
    # Create the base config
    config = SystemConfig(
        system_name="CubeSat",
        concept_name="CubeSat nanosatellite",
        application_domain="spacecraft",
        relationship_type="whole-part",
        api_keys={'openai_api_key': os.getenv('OPENAI_API_KEY')},
        selected_model="gpt-4-turbo-preview",
        ollama_api_key="ollama",
        ollama_api_base="http://localhost:11434/v1",
        ollama_emb_base="http://localhost:11434/api",
        embedding_model= "text-embedding-3-small", #"nomic-embed-text", #"text-embedding-3-small",
        inference_type="llm",
        reference_files={},
        output_directory="../output",
        vectorstore_directory="",
        check_enable_validation=False,
        case="i"
    )
    
    # Your DSM prompt
    # _prompt = """
    # Please identify system-level whole-part relationships between the components of CubeSat (nanosatellite), as listed below, and represent them in a Design Structure Matrix (DSM) format.  \

    # The relationships should be articulated in plain English, capturing the essence of their interactions. \

    # Here is the list of subsystems/components to analyze: ['Spacecraft', 'Guidance, Navigation, and Control (GNC) Subsystem', 'Propulsion Subsystem', 'Power Subsystem', 'Reaction Wheel', 'GNC Software'] \

    # The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
        
    # Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = 6 for this example), with rows and columns labeled accordingly. \

    # Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \

    # If you don't know the answer, just write 2 for the specific cell that you are unsure of. Don't make up an answer. \

    # Provide your output in **Python list format** in the same order as the components in the list above. \
    # """
    
    # Define the prompt
    _prompt = f"""
    Please identify system-level {config.relationship_type} relationships between the components of {config.concept_name}, as listed below, and represent them in a Design Structure Matrix (DSM) format.  \

    The relationships should be articulated in plain English, capturing the essence of their interactions. \

    Here is the list of subsystems/components to analyze: {config.predicted_components} \

    The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
        
    Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = {len(config.predicted_components)} for this example), with rows and columns labeled accordingly. \

    Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \

    If you don't know the answer, just write 2 for the specific cell that you are unsure of. A worst case scenario of the **as a list of lists** shown below (diagonal elements are 1 and rest are 2). Don't make up an answer. \
    
    This is an example of the output format for 7 components (as 7x7 matrix)-based on our system-of-interest the size of the matrix can change. 
    
    final response = [
        [1, 2, 2, 2, 2, 2, 2],
        [2, 1, 2, 2, 2, 2, 2],
        [2, 2, 1, 2, 2, 2, 2],
        [2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 2, 1, 2, 2],
        [2, 2, 2, 2, 2, 1, 2],
        [2, 2, 2, 2, 2, 2, 1]
    ]

    IMPORTANT: Your response must **ONLY** contain a valid **as a list of lists** in the following example format 7x7 matrix. \
    
    The below list of lists is just an example format of your response. **Replace** the values with your **actual analysis** while providing your response with parameter of 'final response'.\
    
    This is an example of the output format for 7 components (as 7x7 matrix)-based on our system-of-interest the size of the matrix can change. \
    
    final response = [
        [1, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 1]
    ]

    IMPORTANT: THE FORMAT OF THE OUTPUT MUST BE AS LIST OF LISTS ONLY SIMILAR TO THE ONE ABOVE (NO LATEX, CODE, OR OTHER ENCODINGS).
    """

    # List of models to test
    models = [
        "gpt-4-turbo-preview", #43.07-42.20=
        # "gpt-4o-2024-11-20", #42.20-42.04=
        # "ollama:mixtral:8x22b_6k",
        # "ollama:llama3.3:70b-8k",
        # "ollama:deepseek-r1:14b-5k"
    ]

    experiment_config = CubeSatExperiment(n_runs=5)
    
    # Run experiments for each model sequentially
    for model in models:
        # At the start
        timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_start = datetime.strptime(timestamp_str, "%Y%m%d-%H%M%S")
        dsm_logger = CustomLogger(config.inference_type,"MultiModelExp",model,timestamp_str)
        results = run_model_experiment(config, model, experiment_config, dsm_logger, _prompt)
        # Save results for this model with timestamp
        output_path = f"../output/experiment_{config.inference_type}_{model}_{timestamp_str}"
        save_model_results(results, config, output_path, logger=dsm_logger, timestamp=timestamp_str)
        timestamp_final_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_end = datetime.strptime(timestamp_final_str, "%Y%m%d-%H%M%S")
        dsm_logger.info(f"Duration of experiment for {model} in {config.inference_type} inference type: {timestamp_end-timestamp_start}")
        dsm_logger.cleanup() # Force flush all handlers
        time.sleep(5) # Give time for the logger to close
        reset_gpus() # Reset GPUs between models
    
    

#### E-iMCLR - Running Multiple Models with Baseline RAG

In [ ]:
# Example usage:
if __name__ == "__main__":
    # Create the base config
    config = SystemConfig(
        system_name="CubeSat",
        concept_name="CubeSat nanosatellite",
        application_domain="spacecraft",
        relationship_type="whole-part",
        api_keys={'openai_api_key': os.getenv('OPENAI_API_KEY')},
        selected_model="gpt-4-turbo-preview",
        ollama_api_key="ollama",
        ollama_api_base="http://localhost:11434/v1",
        ollama_emb_base="http://localhost:11434/api",
        embedding_model= "text-embedding-3-small", #"nomic-embed-text", #"text-embedding-3-small",
        inference_type="llm_rag",
        reference_files={
            'R1': [{
                'path': '../data/use_cases_large/CubeSat/reference_pdfs/R1_combined.pdf'
            }],
            'R2': [{
                'path': '../data/use_cases_large/CubeSat/reference_pdfs/R2_combined.pdf'
            }],
            'R3': [{
                'path': '../data/use_cases_large/CubeSat/reference_pdfs/R3_combined.pdf'
            }]
        },
        output_directory="../output",
        vectorstore_directory="../data/vectorstore"
    )
    
    # Your DSM prompt
    # _prompt = """
    # Please identify system-level whole-part relationships between the components of CubeSat (nanosatellite), as listed below, and represent them in a Design Structure Matrix (DSM) format.  \

    # The relationships should be articulated in plain English, capturing the essence of their interactions. \

    # Here is the list of subsystems/components to analyze: ['Spacecraft', 'Guidance, Navigation, and Control (GNC) Subsystem', 'Propulsion Subsystem', 'Power Subsystem', 'Reaction Wheel', 'GNC Software'] \

    # The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
        
    # Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = 6 for this example), with rows and columns labeled accordingly. \

    # Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \

    # If you don't know the answer, just write 2 for the specific cell that you are unsure of. Don't make up an answer. \

    # Provide your output in **Python list format** in the same order as the components in the list above. \
    # """
    
    # Define the prompt
    _prompt = """
    Please identify system-level proximity relationships (in contact) between the components of CubeSat (nanosatellite), as listed below, and represent them in a Design Structure Matrix (DSM) format.  \

    The relationships should be articulated in plain English, capturing the essence of their interactions. \

    Here is the list of subsystems/components to analyze: ['Spacecraft', 'Guidance, Navigation, and Control (GNC) Subsystem', 'Propulsion Subsystem', 'Power Subsystem', 'Reaction Wheel', 'GNC Software'] \

    The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
        
    Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = 6 for this example), with rows and columns labeled accordingly. \

    Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \

    If you don't know the answer, just write 2 for the specific cell that you are unsure of. A worst case scenario of the **as a list of lists** shown below (diagonal elements are 1 and rest are 2). Don't make up an answer. \
    final response = [
        [1, 2, 2, 2, 2, 2],
        [2, 1, 2, 2, 2, 2],
        [2, 2, 1, 2, 2, 2],
        [2, 2, 2, 1, 2, 2],
        [2, 2, 2, 2, 1, 2],
        [2, 2, 2, 2, 2, 1]
    ]

    IMPORTANT: Your response must **ONLY** contain a valid **as a list of lists** in the following example format 7x7 matrix. \
    
    The below list of lists is just an example format of your response. **Replace** the values with your **actual analysis** while providing your response with parameter of 'final response'.\
    
    final response = [
        [1, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 1]
    ]

    IMPORTANT: THE FORMAT OF THE OUTPUT MUST BE AS LIST OF LISTS ONLY SIMILAR TO THE ONE ABOVE (NO LATEX, CODE, OR OTHER ENCODINGS).
    """

    # List of models to test
    models = [
        "gpt-4-turbo-preview", #43.07-42.20=
        # "gpt-4o-2024-11-20", #42.20-42.04=
        # "ollama:mixtral:8x22b_6k",
        # "ollama:llama3.3:70b-8k",
        # "ollama:deepseek-r1:14b-5k"
    ]

    experiment_config = CubeSatExperiment(n_runs=5)
    
    # Run experiments for each model sequentially
    for model in models:
        # At the start
        timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_start = datetime.strptime(timestamp_str, "%Y%m%d-%H%M%S")
        dsm_logger = CustomLogger(config.inference_type,"MultiModelExp",model,timestamp_str)
        results = run_model_experiment(config, model, experiment_config, dsm_logger, _prompt)
        # Save results for this model with timestamp
        output_path = f"../output/experiment_{config.inference_type}_{model}_{timestamp_str}"
        save_model_results(results, config, output_path, logger=dsm_logger, timestamp=timestamp_str)
        timestamp_final_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_end = datetime.strptime(timestamp_final_str, "%Y%m%d-%H%M%S")
        dsm_logger.info(f"Duration of experiment for {model} in {config.inference_type} inference type: {timestamp_end-timestamp_start}")
        dsm_logger.cleanup() # Force flush all handlers
        time.sleep(5) # Give time for the logger to close
        reset_gpus() # Reset GPUs between models
    

#### E-iMCLR - Running Multiple Models with Baseline GraphRAG

We can use CLI directly to find the DSM. Please make sure that you initialized, autotuned, indexed.
Please change `/path/to/model` with the target model that you initialized, autotuned, and indexed.

In [ ]:
! ./graph_rag_runner.sh query --method local \
                              --query-file "./ragtest/path/to/model/prompts/prompt_cubasat_i.txt" \
                              --root ./ --gpus 0,1,2,3,4,5

### ii: Power Screwdriver

#### E-iiMPL - Running Experiment Baseline LLM

In [ ]:
# Example usage:
if __name__ == "__main__":
    # Create the base config
    config = SystemConfig(
        system_name="PowerDrill",
        concept_name="power screwdriver",
        application_domain="engineering",
        relationship_type="proximity",
        api_keys={'openai_api_key': os.getenv('OPENAI_API_KEY')},
        selected_model="gpt-4-turbo-preview",
        ollama_api_key="ollama",
        ollama_api_base="http://localhost:11434/v1",
        ollama_emb_base="http://localhost:11434/api",
        embedding_model= "text-embedding-3-small", # "nomic-embed-text", #
        inference_type="llm",
        reference_files={},
        output_directory="../output",
        vectorstore_directory="",
        check_enable_validation=True,
        case="ii",
    )

    # Define the prompt
    _prompt = f"""
    Please identify system-level {config.relationship_type} relationships between the components of {config.concept_name}, as listed below, and represent them in a Design Structure Matrix (DSM) format.  \

    The relationships should be articulated in plain English, capturing the essence of their interactions. \

    Here is the list of subsystems/components to analyze: {config.predicted_components} \

    The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
        
    Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = {len(config.predicted_components)} for this example), with rows and columns labeled accordingly. \

    Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \

    If you don't know the answer, just write 2 for the specific cell that you are unsure of. A worst case scenario of the **as a list of lists** shown below (diagonal elements are 1 and rest are 2). Don't make up an answer. \
    
    This is an example of the output format for 7 components (as 7x7 matrix)-based on our system-of-interest the size of the matrix can change. 
    
    final response = [
        [1, 2, 2, 2, 2, 2, 2],
        [2, 1, 2, 2, 2, 2, 2],
        [2, 2, 1, 2, 2, 2, 2],
        [2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 2, 1, 2, 2],
        [2, 2, 2, 2, 2, 1, 2],
        [2, 2, 2, 2, 2, 2, 1]
    ]

    IMPORTANT: Your response must **ONLY** contain a valid **as a list of lists** in the following example format 7x7 matrix. \
    
    The below list of lists is just an example format of your response. **Replace** the values with your **actual analysis** while providing your response with parameter of 'final response'.\
    
    This is an example of the output format for 7 components (as 7x7 matrix)-based on our system-of-interest the size of the matrix can change. \
    
    final response = [
        [1, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 1]
    ]

    IMPORTANT: THE FORMAT OF THE OUTPUT MUST BE AS LIST OF LISTS ONLY SIMILAR TO THE ONE ABOVE (NO LATEX, CODE, OR OTHER ENCODINGS).
    """
    
    # List of models to test
    models = [
        "gpt-4-turbo-preview", #43.07-42.20=
        # "gpt-4o-2024-11-20", #42.20-42.04=
        # "mixtral:8x22b-6k",
        # "llama3.3:70b-8k",
        # "deepseek-r1:14b-5k"
    ]

    experiment_config = PowerDrillExperiment(n_runs=5)
    
    # Run experiments for each model sequentially
    for model in models:
        # At the start
        timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_start = datetime.strptime(timestamp_str, "%Y%m%d-%H%M%S")
        dsm_logger = CustomLogger(config.inference_type,"MultiModelExp",model,timestamp_str)
        results = run_model_experiment(config, model, experiment_config, dsm_logger, _prompt)
        # Save results for this model with timestamp
        output_path = f"../output/experiment_{config.inference_type}_{model}_{timestamp_str}"
        save_model_results(results, config, output_path, logger=dsm_logger, timestamp=timestamp_str)
        timestamp_final_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_end = datetime.strptime(timestamp_final_str, "%Y%m%d-%H%M%S")
        dsm_logger.info(f"Duration of experiment for {model} in {config.inference_type} inference type: {timestamp_end-timestamp_start}")
        dsm_logger.cleanup() # Force flush all handlers
        time.sleep(5) # Give time for the logger to close
        reset_gpus() # Reset GPUs between models

#### E-iiMPLR - Running Multiple Models with Baseline RAG

In [ ]:
if __name__ == "__main__":
    # Create the base config
    config = SystemConfig(
        system_name="power screwdriver",
        concept_name="power screwdriver",
        application_domain="engineering",
        relationship_type="proximity",
        api_keys={'openai_api_key': os.getenv('OPENAI_API_KEY')},
        selected_model="gpt-4-turbo-preview",
        ollama_api_key="ollama",
        ollama_api_base="http://localhost:11434/v1",
        ollama_emb_base="http://localhost:11434/api",
        embedding_model= "text-embedding-3-small", # "nomic-embed-text", #
        inference_type="llm_rag",
        reference_files={
            'R1': [{
                'path': '../data/use_cases_large/PowerDrill/reference_pdfs/R1_combined.pdf'
            }],
            'R2': [{
                'path': '../data/use_cases_large/PowerDrill/reference_pdfs/[2012 Eppinger] R2-Design structure matrix methods and applications.pdf'
            }],
            'R3': [{
                'path': '../data/use_cases_large/PowerDrill/reference_pdfs/R3_combined.pdf'
            }]
        },
        output_directory="../output",
        vectorstore_directory="",
        check_enable_validation=True,
        case="ii",
    )

    # Define the prompt
    _prompt = f"""
    Please identify system-level {config.relationship_type} relationships between the components of {config.concept_name}, as listed below, and represent them in a Design Structure Matrix (DSM) format.  \

    The relationships should be articulated in plain English, capturing the essence of their interactions. \

    Here is the list of subsystems/components to analyze: {config.predicted_components} \

    The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
        
    Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = {len(config.predicted_components)} for this example), with rows and columns labeled accordingly. \

    Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \

    If you don't know the answer, just write 2 for the specific cell that you are unsure of. A worst case scenario of the **as a list of lists** shown below (diagonal elements are 1 and rest are 2). Don't make up an answer. \
    
    This is an example of the output format for 7 components (as 7x7 matrix)-based on our system-of-interest the size of the matrix can change. 
    
    final response = [
        [1, 2, 2, 2, 2, 2, 2],
        [2, 1, 2, 2, 2, 2, 2],
        [2, 2, 1, 2, 2, 2, 2],
        [2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 2, 1, 2, 2],
        [2, 2, 2, 2, 2, 1, 2],
        [2, 2, 2, 2, 2, 2, 1]
    ]

    IMPORTANT: Your response must **ONLY** contain a valid **as a list of lists** in the following example format 7x7 matrix. \
    
    The below list of lists is just an example format of your response. **Replace** the values with your **actual analysis** while providing your response with parameter of 'final response'.\
    
    This is an example of the output format for 7 components (as 7x7 matrix)-based on our system-of-interest the size of the matrix can change. \
    
    final response = [
        [1, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 1]
    ]

    IMPORTANT: THE FORMAT OF THE OUTPUT MUST BE AS LIST OF LISTS ONLY SIMILAR TO THE ONE ABOVE (NO LATEX, CODE, OR OTHER ENCODINGS).
    """
    
    # List of models to test
    models = [
        "gpt-4-turbo-preview", #43.07-42.20=
        # "gpt-4o-2024-11-20", #42.20-42.04=
        # "mixtral:8x22b-6k",
        # "llama3.3:70b-8k",
        # "deepseek-r1:14b-5k"
    ]

    experiment_config = PowerDrillExperiment(n_runs=5)
    
    # Run experiments for each model sequentially
    for model in models:
        # At the start
        timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_start = datetime.strptime(timestamp_str, "%Y%m%d-%H%M%S")
        dsm_logger = CustomLogger(config.inference_type,"MultiModelExp",model,timestamp_str)
        results = run_model_experiment(config, model, experiment_config, dsm_logger, _prompt)
        # Save results for this model with timestamp
        output_path = f"../output/experiment_{config.inference_type}_{model}_{timestamp_str}"
        save_model_results(results, config, output_path, logger=dsm_logger, timestamp=timestamp_str)
        timestamp_final_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_end = datetime.strptime(timestamp_final_str, "%Y%m%d-%H%M%S")
        dsm_logger.info(f"Duration of experiment for {model} in {config.inference_type} inference type: {timestamp_end-timestamp_start}")
        dsm_logger.cleanup() # Force flush all handlers
        time.sleep(5) # Give time for the logger to close
        reset_gpus() # Reset GPUs between models

#### E-iiMPLR - Running Multiple Models with Baseline GraphRAG

In [10]:
# Assume necessary imports like os, datetime, time, SystemConfig, etc.
# Assume PowerDrillExperiment, CustomLogger, reset_gpus are defined/imported

if __name__ == "__main__":
    # Create the base config
    config = SystemConfig(
        system_name="Power screwdriver",
        concept_name="power screwdriver",
        application_domain="engineering",
        relationship_type="proximity",
        api_keys={'openai_api_key': os.getenv('OPENAI_API_KEY')},
        selected_model="gpt-4-turbo-preview",
        ollama_api_key="ollama",
        ollama_api_base="http://localhost:11434/v1",
        ollama_emb_base="http://localhost:11434/api",
        embedding_model= "text-embedding-3-small", # "nomic-embed-text", #
        inference_type="llm_rag_graph",
        reference_files={},
        reference_types=["r1_r2"], #"r2_r3"
        output_directory="../output",
        vectorstore_directory="",
        check_enable_validation=True,
        graphrag_input_dir="../data/input/power_screwdriver", # !IMPORTANT: Path to docs for GraphRAG indexing
        case="ii",
    )

    # Define the prompt
    _prompt = f"""
    Please identify system-level {config.relationship_type} relationships between the components of {config.concept_name}, as listed below, and represent them in a Design Structure Matrix (DSM) format.  \

    The relationships should be articulated in plain English, capturing the essence of their interactions. \

    Here is the list of subsystems/components to analyze: {config.predicted_components} \

    The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
        
    Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = {len(config.predicted_components)} for this example), with rows and columns labeled accordingly. \

    Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \

    If you don't know the answer, just write 2 for the specific cell that you are unsure of. A worst case scenario of the **as a list of lists** shown below (diagonal elements are 1 and rest are 2). Don't make up an answer. \
    
    This is an example of the output format for 7 components (as 7x7 matrix)-based on our system-of-interest the size of the matrix can change. 
    
    final response = [
        [1, 2, 2, 2, 2, 2, 2],
        [2, 1, 2, 2, 2, 2, 2],
        [2, 2, 1, 2, 2, 2, 2],
        [2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 2, 1, 2, 2],
        [2, 2, 2, 2, 2, 1, 2],
        [2, 2, 2, 2, 2, 2, 1]
    ]

    IMPORTANT: Your response must **ONLY** contain a valid **as a list of lists** in the following example format 7x7 matrix. \
    
    The below list of lists is just an example format of your response. **Replace** the values with your **actual analysis** while providing your response with parameter of 'final response'.\
    
    This is an example of the output format for 7 components (as 7x7 matrix)-based on our system-of-interest the size of the matrix can change. \
    
    final response = [
        [1, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 1]
    ]

    IMPORTANT: THE FORMAT OF THE OUTPUT MUST BE AS LIST OF LISTS ONLY SIMILAR TO THE ONE ABOVE (NO LATEX, CODE, OR OTHER ENCODINGS).
    """
    
    # List of models to test
    models = [
        # "gpt-4-turbo-preview", #43.07-42.20=
          "ollama:mixtral:8x22b-6k",
        # "ollama:llama3.3:70b-8k",
        # "ollama:deepseek-r1:14b-5k"
    ]

    experiment_config = PowerDrillExperiment(n_runs=5)
 
     
    # Run experiments for each model sequentially
    for model in models:
        # At the start
        timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_start = datetime.strptime(timestamp_str, "%Y%m%d-%H%M%S")
        dsm_logger = CustomLogger(config.inference_type,"MultiModelExp",model,timestamp_str)
        results = run_model_experiment(config, model, experiment_config, dsm_logger, _prompt)
        # Save results for this model with timestamp
        output_path = f"../output/experiment_{config.inference_type}_{model}_{timestamp_str}"
        save_model_results(results, config, output_path, logger=dsm_logger, timestamp=timestamp_str)
        timestamp_final_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_end = datetime.strptime(timestamp_final_str, "%Y%m%d-%H%M%S")
        dsm_logger.info(f"Duration of experiment for {model} in {config.inference_type} inference type: {timestamp_end-timestamp_start}")
        dsm_logger.cleanup() # Force flush all handlers
        time.sleep(5) # Give time for the logger to close
        reset_gpus() # Reset GPUs between models

2025-04-10 21:02:00,249 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Starting experiment for model: ollama:mixtral:8x22b-6k at 2025-04-10 21:02:00.249365
2025-04-10 21:02:00,250 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Inference type: llm_rag_graph
2025-04-10 21:02:00,251 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - GraphRAG mode: Will run on the single indexed graph.
2025-04-10 21:02:00,251 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - --- Starting Combination: ['indexed_graph'] ---
2025-04-10 21:02:00,253 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - --- Run 1/5 for combo ['indexed_graph'] ---
2025-04-10 21:02:00,285 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - DEBUG: Initializing GraphRAGInference.
2025-04-10 21:02:00,286 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Model name formatted: auto_mixtral_8x22b_6k_r1_r2-te3s
2025-04-1

INFO: Initializing LLM for GraphRAG. Model: 'ollama:mixtral:8x22b-6k', Type: 'llm_rag_graph'
INFO: Attempting GraphRAGChatOpenAI for Ollama model: mixtral:8x22b-6k at http://localhost:11434/v1


2025-04-10 21:02:00,528 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Indexer data processed.
2025-04-10 21:02:00,528 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Initializing tokenizer and embedder...
2025-04-10 21:02:00,703 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Tokenizer and embedder initialized.
2025-04-10 21:02:00,704 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Initializing vector store...
2025-04-10 21:02:00,742 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Initializing context builder...
2025-04-10 21:02:00,744 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Context builder initialized.
2025-04-10 21:02:00,744 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Context Params: {'text_unit_prop': 0.5, 'community_prop': 0.1, 'conversation_history_max_turns': 5, 'conversation_history_user_turns_only': True, 'top_k_mapped_entities': 10,

INFO: Initializing LLM for GraphRAG. Model: 'ollama:mixtral:8x22b-6k', Type: 'llm_rag_graph'
INFO: Attempting GraphRAGChatOpenAI for Ollama model: mixtral:8x22b-6k at http://localhost:11434/v1


2025-04-10 21:03:20,828 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Indexer data processed.
2025-04-10 21:03:20,829 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Initializing tokenizer and embedder...
2025-04-10 21:03:20,845 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Tokenizer and embedder initialized.
2025-04-10 21:03:20,846 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Initializing vector store...
2025-04-10 21:03:20,850 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Initializing context builder...
2025-04-10 21:03:20,851 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Context builder initialized.
2025-04-10 21:03:20,851 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Context Params: {'text_unit_prop': 0.5, 'community_prop': 0.1, 'conversation_history_max_turns': 5, 'conversation_history_user_turns_only': True, 'top_k_mapped_entities': 10,

...

INFO: Initializing LLM for GraphRAG. Model: 'ollama:mixtral:8x22b-6k', Type: 'llm_rag_graph'
INFO: Attempting GraphRAGChatOpenAI for Ollama model: mixtral:8x22b-6k at http://localhost:11434/v1


2025-04-10 21:04:58,633 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Indexer data processed.
2025-04-10 21:04:58,634 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Initializing tokenizer and embedder...
2025-04-10 21:04:58,650 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Tokenizer and embedder initialized.
2025-04-10 21:04:58,651 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Initializing vector store...
2025-04-10 21:04:58,654 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Initializing context builder...
2025-04-10 21:04:58,655 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Context builder initialized.
2025-04-10 21:04:58,656 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Context Params: {'text_unit_prop': 0.5, 'community_prop': 0.1, 'conversation_history_max_turns': 5, 'conversation_history_user_turns_only': True, 'top_k_mapped_entities': 10,

INFO: Initializing LLM for GraphRAG. Model: 'ollama:mixtral:8x22b-6k', Type: 'llm_rag_graph'
INFO: Attempting GraphRAGChatOpenAI for Ollama model: mixtral:8x22b-6k at http://localhost:11434/v1


2025-04-10 21:05:47,500 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Indexer data processed.
2025-04-10 21:05:47,501 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Initializing tokenizer and embedder...
2025-04-10 21:05:47,517 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Tokenizer and embedder initialized.
2025-04-10 21:05:47,518 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Initializing vector store...
2025-04-10 21:05:47,521 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Initializing context builder...
2025-04-10 21:05:47,522 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Context builder initialized.
2025-04-10 21:05:47,523 - DSM_MultiModelExp_llm_rag_graph_ollama:mixtral:8x22b-6k - INFO - Context Params: {'text_unit_prop': 0.5, 'community_prop': 0.1, 'conversation_history_max_turns': 5, 'conversation_history_user_turns_only': True, 'top_k_mapped_entities': 10,


Resetting GPUs...
GPUs reset successfully
Waiting for 15 seconds after GPU reset...


### ii: CubeSat

#### E-iMCL - Running Multiple Models with Baseline LLM

In [ ]:
# Example usage:
if __name__ == "__main__":
    # Create the base config
    config = SystemConfig(
        system_name="CubeSat",                          # MUST BE CHANGED BASED ON THE USE CASE
        concept_name="CubeSat (nanosatellite)",         # MUST BE CHANGED BASED ON THE USE CASE
        application_domain="engineering",               # MUST BE CHANGED BASED ON THE USE CASE
        relationship_type="hierarchical whole-part",    # MUST BE CHANGED BASED ON THE USE CASE
        api_keys={'openai_api_key': os.getenv('OPENAI_API_KEY')},   # MUST BE CHANGED BASED ON THE MODEL
        selected_model="ollama:deepseek-r1:14b-5k",                 # MUST BE CHANGED BASED ON THE MODEL
        ollama_api_key="ollama",                                    # MUST BE CHANGED BASED ON THE MODEL    
        ollama_api_base="http://localhost:11434/v1",                # MUST BE CHANGED BASED ON THE MODEL
        ollama_emb_base="http://localhost:11434/api",               # MUST BE CHANGED BASED ON THE MODEL   
        embedding_model= "text-embedding-3-small", # "nomic-embed-text", #
        inference_type="llm",                                       # MUST BE CHANGED BASED ON THE METHOD 
        reference_files={},                                         # MUST BE CHANGED BASED ON THE METHOD
        output_directory="../output",
        vectorstore_directory="",
        check_enable_validation=True,                               # Enable or Disable LLM-based Validation
        case="ii",                                                  # case: Identify + Classify
    )

    # Define the prompt
    _prompt = f"""
    Please identify system-level {config.relationship_type} relationships between the components of {config.concept_name}, as listed below, and represent them in a Design Structure Matrix (DSM) format.  \
    
    The relationships should be articulated in plain English, capturing the essence of their interactions. \
    
    Here is the list of subsystems/components to analyze: {config.predicted_components} \
    
    The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
    
    Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = {len(config.predicted_components)} for this example), with rows and columns labeled accordingly. \
    
    Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \
    
    If you don't know the answer, just write 2 for the specific cell that you are unsure of. A worst case scenario of the **as a list of lists** shown below (diagonal elements are 1 and unsure ones are 2). Don't make up an answer. \
    
    final response = [[1, 2, 2, 2, 2, 2, 2],[2, 1, 2, 2, 2, 2, 2],[2, 2, 1, 2, 2, 2, 2],[2, 2, 2, 1, 2, 2, 2],[2, 2, 2, 2, 1, 2, 2],[2, 2, 2, 2, 2, 1, 2],[2, 2, 2, 2, 2, 2, 1]] for 7 components \
    
    final response = [[1, 2, 2],[2, 1, 2],[2, 2, 1]] for 3 components \
    
    IMPORTANT: Your response must **ONLY** contain a valid **as a list of lists** in the following example format 7x7 matrix. \
    
    The below list of lists is just an example format of a response. **Replace** the values with your **actual analysis** while providing your response with parameter of 'final response'.\
    
    final response = [[1, 0, 0, 0, 0, 0, 0],[0, 1, 0, 0, 0, 0, 0],[0, 0, 1, 0, 0, 0, 0],   [0, 0, 0, 1, 0, 0, 0],[0, 0, 0, 0, 1, 0, 0],[0, 0, 0, 0, 0, 1, 0],[0, 0, 0, 0, 0, 0, 1]] for 7 components \
    
    final response = [[1, 0, 0],[0, 1, 0],[0, 0, 1]] for 3 components \
     
    IMPORTANT: THE FORMAT OF THE OUTPUT MUST BE AS LIST OF LISTS ONLY SIMILAR TO THE ONE ABOVE (NO LATEX, CODE, OR OTHER ENCODINGS).
    """
    
    # List of models to test
    models = [
        "gpt-4-turbo-preview", #43.07-42.20=
        "gpt-4o-2024-11-20", #42.20-42.04=
        "ollama:mixtral:8x22b_6k",
        "ollama:llama3.3:70b-8k",
        "ollama:deepseek-r1:14b-5k"
    ]

    experiment_config = CubeSatExperiment(n_runs=5) # MUST 
    
    # Run experiments for each model sequentially
    for model in models:
        # At the start
        timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_start = datetime.strptime(timestamp_str, "%Y%m%d-%H%M%S")
        dsm_logger = CustomLogger(config.inference_type,"MultiModelExp",model,timestamp_str)
        results = run_model_experiment(config, model, experiment_config, dsm_logger, _prompt)
        # Save results for this model with timestamp
        output_path = f"../output/experiment_{config.inference_type}_{model}_{timestamp_str}"
        save_model_results(results, config, output_path, logger=dsm_logger, timestamp=timestamp_str)
        timestamp_final_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_end = datetime.strptime(timestamp_final_str, "%Y%m%d-%H%M%S")
        dsm_logger.info(f"Duration of experiment for {model} in {config.inference_type} inference type: {timestamp_end-timestamp_start}")
        dsm_logger.cleanup() # Force flush all handlers
        time.sleep(5) # Give time for the logger to close
        reset_gpus() # Reset GPUs between models

#### E-iiMCLR - Running Multiple Models with Baseline RAG

In [ ]:
# For the experiments you need to change concept_name, system_name, application_domain, relationship_type, inference_type, reference_files, case, the class for the 
# experiment config, and target models (uncomment whatever is needed for OAI please add your api_key).

if __name__ == "__main__":
    # Create the base config
    config = SystemConfig(
        concept_name="CubeSat (nanosatellite)",
        system_name="CubeSat",
        application_domain="engineering",
        relationship_type="hierarchical whole-part",
        api_keys={'openai_api_key': os.getenv('OPENAI_API_KEY')},
        selected_model="ollama:deepseek-r1:14b-5k",
        ollama_api_key="ollama",
        ollama_api_base="http://localhost:11434/v1",
        ollama_emb_base="http://localhost:11434/api",
        embedding_model= "text-embedding-3-small", # "nomic-embed-text", #
        inference_type="llm_rag",
        reference_files={
            'R1': [{
                'path': '../data/use_cases_large/CubeSat/reference_pdfs/R1_combined.pdf'
            }],
            'R2': [{
                'path': '../data/use_cases_large/CubeSat/reference_pdfs/R2_combined.pdf'
            }],
            'R3': [{
                'path': '../data/use_cases_large/CubeSat/reference_pdfs/R3_combined.pdf'
            }]
        },
        output_directory="../output",
        vectorstore_directory="",
        check_enable_validation=True,
        case="ii",
        predicted_components="",
    )


    # Define the prompt
    _prompt = f"""
    Please identify system-level {config.relationship_type} relationships between the components of {config.concept_name}, as listed below, and represent them in a Design Structure Matrix (DSM) format.  \
    
    The relationships should be articulated in plain English, capturing the essence of their interactions. \
    
    Here is the list of subsystems/components to analyze: {config.predicted_components} \
    
    The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
    
    Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = {len(config.predicted_components)} for this example), with rows and columns labeled accordingly. \
    
    Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \
    
    If you don't know the answer, just write 2 for the specific cell that you are unsure of. A worst case scenario of the **as a list of lists** shown below (diagonal elements are 1 and unsure ones are 2). Don't make up an answer. \
    
    final response = [[1, 2, 2, 2, 2, 2, 2],[2, 1, 2, 2, 2, 2, 2],[2, 2, 1, 2, 2, 2, 2],[2, 2, 2, 1, 2, 2, 2],[2, 2, 2, 2, 1, 2, 2],[2, 2, 2, 2, 2, 1, 2],[2, 2, 2, 2, 2, 2, 1]] for 7 components \
    
    final response = [[1, 2, 2],[2, 1, 2],[2, 2, 1]] for 3 components \
    
    IMPORTANT: Your response must **ONLY** contain a valid **as a list of lists** in the following example format 7x7 matrix (NO LATEX, NO CODE, NO DESCRIPTION, OR OTHER ENCODINGS, JUST THE LIST OF LISTS). \
    
    The below list of lists is just examples format of a response. **Replace** the values with your **actual analysis** while providing your response with parameter of 'final response'.\
    
    final response = [[1, 0, 0, 0, 0, 0, 0],[0, 1, 0, 0, 0, 0, 0],[0, 0, 1, 0, 0, 0, 0],   [0, 0, 0, 1, 0, 0, 0],[0, 0, 0, 0, 1, 0, 0],[0, 0, 0, 0, 0, 1, 0],[0, 0, 0, 0, 0, 0, 1]] for 7 components \
    
    final response = [[1, 0, 0],[0, 1, 0],[0, 0, 1]] for 3 components \
     
    IMPORTANT: THE FORMAT OF THE OUTPUT MUST BE AS LIST OF LISTS ONLY SIMILAR TO THE ONE ABOVE (NO LATEX, NO CODE, NO DESCRIPTION, OR OTHER ENCODINGS, JUST THE LIST OF LISTS). \
    """
    
    # List of models to test
    models = [
        # "gpt-4-turbo-preview", #43.07-42.20=
        # "gpt-4o-2024-11-20", #42.20-42.04=
        # "ollama:mixtral:8x22b_6k",
        # "ollama:llama3.3:70b-8k",
        "ollama:deepseek-r1:14b-5k"
    ]

    experiment_config = CubeSatExperiment(n_runs=5)
    
    # Run experiments for each model sequentially
    for model in models:
        # At the start
        timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_start = datetime.strptime(timestamp_str, "%Y%m%d-%H%M%S")
        dsm_logger = CustomLogger(config.inference_type,"MultiModelExp",model,timestamp_str)
        results = run_model_experiment(config, model, experiment_config, dsm_logger, _prompt)
        # Save results for this model with timestamp
        output_path = f"../output/experiment_{config.inference_type}_{model}_{timestamp_str}"
        save_model_results(results, config, output_path, logger=dsm_logger, timestamp=timestamp_str)
        timestamp_final_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_end = datetime.strptime(timestamp_final_str, "%Y%m%d-%H%M%S")
        dsm_logger.info(f"Duration of experiment for {model} in {config.inference_type} inference type: {timestamp_end-timestamp_start}")
        dsm_logger.cleanup() # Force flush all handlers
        time.sleep(5) # Give time for the logger to close
        reset_gpus() # Reset GPUs between models

#### E-iiMCLR - Running Multiple Models with Baseline GraphRAG

In [ ]:
# Assume necessary imports like os, datetime, time, SystemConfig, etc.
# Assume PowerDrillExperiment, CustomLogger, reset_gpus are defined/imported

if __name__ == "__main__":
    # Create the base config
    config = SystemConfig(
        concept_name="CubeSat (nanosatellite)",
        system_name="CubeSat",
        application_domain="engineering",
        relationship_type="hierarchical whole-part",
        api_keys={'openai_api_key': os.getenv('OPENAI_API_KEY')},
        selected_model="gpt-4-turbo-preview",
        ollama_api_key="ollama",
        ollama_api_base="http://localhost:11434/v1",
        ollama_emb_base="http://localhost:11434/api",
        embedding_model= "text-embedding-3-small", # "nomic-embed-text", #
        inference_type="llm_rag_graph",
        reference_files={},
        reference_types=["r2", "r3"], 
        output_directory="../output",
        vectorstore_directory="",
        check_enable_validation=True,
        graphrag_input_dir="../data/input/cubesat", # !IMPORTANT: Path to docs for GraphRAG indexing
        case="ii",
    )

    # Define the prompt
    _prompt = f"""
    Please identify system-level {config.relationship_type} relationships between the components of {config.concept_name}, as listed below, and represent them in a Design Structure Matrix (DSM) format.  \

    The relationships should be articulated in plain English, capturing the essence of their interactions. \

    Here is the list of subsystems/components to analyze: {config.predicted_components} \

    The DSM should clearly indicate the interactions between these components as 1 exist and 0 does not exist, considering both the presence and nature of these connections. \
        
    Present the output in a square matrix format (number of rows and columns MUST be the same as the number of components = {len(config.predicted_components)} for this example), with rows and columns labeled accordingly. \

    Diagonal elements should be 1 in DSM matrix since each component interacts with itself. \

    If you don't know the answer, just write 2 for the specific cell that you are unsure of. A worst case scenario of the **as a list of lists** shown below (diagonal elements are 1 and rest are 2). Don't make up an answer. \
    
    This is an example of the output format for 7 components (as 7x7 matrix)-based on our system-of-interest the size of the matrix can change. 
    
    final response = [
        [1, 2, 2, 2, 2, 2, 2],
        [2, 1, 2, 2, 2, 2, 2],
        [2, 2, 1, 2, 2, 2, 2],
        [2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 2, 1, 2, 2],
        [2, 2, 2, 2, 2, 1, 2],
        [2, 2, 2, 2, 2, 2, 1]
    ]

    IMPORTANT: Your response must **ONLY** contain a valid **as a list of lists** in the following example format 7x7 matrix. \
    
    The below list of lists is just an example format of your response. **Replace** the values with your **actual analysis** while providing your response with parameter of 'final response'.\
    
    This is an example of the output format for 7 components (as 7x7 matrix)-based on our system-of-interest the size of the matrix can change. \
    
    final response = [
        [1, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 1]
    ]

    IMPORTANT: THE FORMAT OF THE OUTPUT MUST BE AS LIST OF LISTS ONLY SIMILAR TO THE ONE ABOVE (NO LATEX, CODE, OR OTHER ENCODINGS).
    """
    
    # List of models to test
    models = [
        "gpt-4-turbo-preview", #43.07-42.20=
        # "ollama:mixtral:8x22b-6k",
        # "ollama:llama3.3:70b-8k",
        # "ollama:deepseek-r1:14b-5k"
    ]

    experiment_config = CubeSatExperiment(n_runs=5)
 
    # Run experiments for each model sequentially
    for model in models:
        # At the start
        timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_start = datetime.strptime(timestamp_str, "%Y%m%d-%H%M%S")
        dsm_logger = CustomLogger(config.inference_type,"MultiModelExp",model,timestamp_str)
        results = run_model_experiment(config, model, experiment_config, dsm_logger, _prompt)
        # Save results for this model with timestamp
        output_path = f"../output/experiment_{config.inference_type}_{model}_{timestamp_str}"
        save_model_results(results, config, output_path, logger=dsm_logger, timestamp=timestamp_str)
        timestamp_final_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        timestamp_end = datetime.strptime(timestamp_final_str, "%Y%m%d-%H%M%S")
        dsm_logger.info(f"Duration of experiment for {model} in {config.inference_type} inference type: {timestamp_end-timestamp_start}")
        dsm_logger.cleanup() # Force flush all handlers
        time.sleep(15) # Give time for the logger to close
        reset_gpus() # Reset GPUs between models

### Run Experiment from JSON

In [4]:
def run_experiments(experiment_numbers: list[str], 
                    api_key: str,
                    base_path: str,
                    exclude_patterns: list = None
                    ):
    """
    Run specific experiments while excluding certain patterns
    
    Args:
        experiment_numbers: list of strings, e.g., ["2", "3", "5"] for experiments 2, 3, and 5
        api_key: API key for the LLM service
        exclude_patterns: list of strings to exclude, e.g., ['graph', 'rag']
    """    
    experiment_files = []
    for exp_num in experiment_numbers:
        # Match exact experiment number using regex pattern
        pattern = f'*experiment_{exp_num}_*.json'
        matching_files = glob.glob(os.path.join(base_path, pattern))
        experiment_files.extend(matching_files)
    
    # Apply exclusions
    if exclude_patterns:
        for pattern in exclude_patterns:
            experiment_files = [f for f in experiment_files if pattern.lower() not in f.lower()]
    
    print(f"Found {len(experiment_files)} experiments to run:")
    for f in experiment_files:
        print(f"- {os.path.basename(f)}")
    
    for config_path in experiment_files:
        try:
            time.sleep(10)
            print("-" * 50)
            print(f"Starting experiment: {os.path.basename(config_path)} at {datetime.now()}")
            
            # Create config and initialize DSM
            config = SystemConfig.from_json(config_path)
            config.set_api_key(api_key)  # Use the provided API key

            # if 'rag' in config.inference_type.lower():
            #     print(f"Using existing vectorstore for RAG experiment: {config.system_name}")

            time.sleep(5)
            # Initialize DSM
            dsm = DesignStructureMatrix(config)
            time.sleep(15)  # Sleep between initialization and component identification
            
            # Identify components
            components = dsm.identify_components()
            dsm.log_experiment(f"Identified components: {components}")
            
            time.sleep(15)  # Longer sleep before relationship analysis
            
            # Create and analyze the DSM
            matrix = dsm.analyze_component_relationships()
            dsm.log_experiment(f"DSM matrix:\n{matrix}")
            
            # Save results
            dsm.save_results()
            dsm.log_experiment("Results saved successfully")
            
            time.sleep(10)  # Longer sleep between experiments
            
            print(f"Completed experiment: {os.path.basename(config_path)} at {datetime.now()}\n")
            
        except Exception as e:
            print(f"Error in experiment {os.path.basename(config_path)}: {str(e)}")
            continue  # Continue with next experiment if one fails
        finally:
            # Clean up logging handlers
            root_logger = logging.getLogger()
            for handler in root_logger.handlers[:]:
                handler.close()
                root_logger.removeHandler(handler)
            print("-" * 50)

In [ ]:
# Path for config file
config_path = '../json/experiments/CuttingSystem/CuttingSystem_experiment_4_llm.json'

# Create config and initialize DSM
config = SystemConfig.from_json(config_path)
config.set_api_key('<Your API Key>') # Allows us to enter API key instead of storing them

dsm = DesignStructureMatrix(config)

# Identify components (Please wait a bit after running previous cell)
time.sleep(20)
components = dsm.identify_components()
dsm.log_experiment(f"Identified components: {components}")

# Create and analyze the DSM
time.sleep(20)
matrix = dsm.analyze_component_relationships()
dsm.log_experiment("Analysis complete.")
# Save results
dsm.save_results()
dsm.log_experiment('Results saved successfully.')


- For running multiple experiments based on the json information.

In [ ]:
# Example usage:
# run_experiments(
#     experiment_numbers=["2", "3", "5"],  # Run experiments 2, 3, and 5
#     exclude_patterns=["graph", "rag"]    # exclude any files containing these terms
# )

# Base path for experiments (recall that we are in ./scripts/)
base_path = '../json/experiments/PowerDrill'

# Example usage
api_key = os.getenv('OPENAI_API_KEY')  # Get API key from environment variable

# Run experiments
run_experiments(
    experiment_numbers=["1","2","4","5","7","8"], #,"5","6","8","9"],
    api_key=api_key,
    base_path = base_path,
    exclude_patterns=["graph","analysis"] # analysis added to ignore analysis json files
)

## ====== Metrics in Step 3 while using the results from here.======>